# Structural Break (Real-Time) — pivotal feature bank + LightGBM

Design principles baked into this notebook:

1. **TS-AUC compares series cross-sectionally at a fixed `t`, but inference is per-series.**
   So every evidence feature must be *pivotal*: same null distribution regardless of the
   underlying DGP. We get this by transforming every online point through the **historical
   empirical CDF** (PIT), then through `Phi^-1`. Under "no break", `z_t ~ N(0,1)` for every
   series, whatever its marginal is.

2. **The AUC-optimal score is any monotone transform of `P(break already happened | data)`.**
   So we train a plain binary classifier (logloss) and use its probability directly.
   A pure function of `t` is a constant within a t-slice => `t` is a free feature that can
   only help through interactions.

3. **Feature bank = (what changed) x (how to aggregate under unknown tau).**
   - what changed: location / scale / shape / tails / dependence / trend
   - how to aggregate: expanding, EWMA (3 decays), trailing windows (4 lengths),
     Page-CUSUM (3 drifts).  Multi-scale replaces the O(t) "max over candidate tau".

4. **Two channels.** The same bank is run on
   - `z`  = PIT-normal of the raw value        -> marginal / distributional change
   - `ze` = PIT-normal of the AR(p) residual   -> dependence / dynamics change

5. **Context features** (historical kurtosis, AR coefs, sd/MAD, n_H) carry no break evidence.
   They let LightGBM learn *how much to trust* each evidence feature for this kind of process.

Everything is O(1) per online step and the *same* extractor object is used by `train()` and
`infer()`, so there is no train/serve skew.


## Setup

Paste your own token below.


In [61]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

#import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [62]:
%pip install crunch-cli --upgrade --quiet --progress-bar off
!crunch setup-notebook structural-break-real-time <YOUR_CRUNCH_TOKEN>

Note: you may need to restart the kernel to use updated packages.
crunch-cli, version 12.0.1
delete /kaggle/working/.crunchdao
you appear to have never submitted code before
data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test.reduced.parquet (106299 bytes)
data/y_test_index.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test_index.reduced.parquet (3674 bytes)

In [63]:
import bisect
import copy
import math
import os
from typing import Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import special as _sp

def ndtri(x):                 # plain-Python wrapper: scipy's ufunc has __module__ = None,
    return _sp.ndtri(x)       # so cloudpickle would mis-resolve it to __mp_main__ in a
                              # loky worker. The wrapper pickles by value and carries a
                              # proper module reference to scipy.special instead.
from sklearn.metrics import roc_auc_score


In [64]:
import crunch

crunch_tools = crunch.load_notebook()


loaded crunch tools for module: <module '__main__'>

cli version: 12.0.1
available ram: 31.35 gb
available cpu: 4 core
----


In [65]:
train_data, test_data = crunch_tools.load_data()
print("train series:", len(train_data))


data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test.reduced.parquet (106299 bytes)
data/y_test.reduced.parquet: already exists, file length match
data/y_test_index.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws

## 1. The feature engine

`HistoricalRef` does the one-off O(n_H) work on the historical segment:
sorted array + `Phi^-1` lookup table for the PIT, robust moments, AR(p) coefficients by
Yule-Walker, the residual PIT table, and (new) the deterministic PIT-normal quantile grids
`z_ref`/`e_ref` used as the fixed "historical sample" side of every Tier 1-4 two-sample test.

**Feature registry, not a hand-maintained list.** A `FeatureFamily` is one self-contained
statistic (or small group of related statistics): it declares its own feature names, does any
one-off historical setup, and knows how to turn the shared per-step state into its own output
vector. A module-level `REGISTRY: list[FeatureFamily]` is the single source of truth --
`FEATURES` and `N_FEATURES` are *derived* from it, never hand-written, and `StreamFeatures`
is a thin driver that loops the registry and asserts each family returns exactly as many
values as `family.names()` promises. Adding a feature is: write one class, append it to
`REGISTRY`. Nothing else changes.

Several existing blocks (the recent-vs-previous contrast, the local-trend slope) read the
window running-sums that the per-channel moment bank also emits from -- so those trackers
(ring buffer, window sums, EWMA, CUSUM state) are pulled out into a shared `ChannelState`,
pushed once per step by the driver, and any family needing a channel reads it off a per-step
`Ctx` instead of recomputing it.

To add a feature: write a `FeatureFamily` subclass and append an instance to `REGISTRY`
(section 1d/1f-1i below). No positional-agreement assertion to keep in sync by hand -- the
mismatch check is per-family and fails immediately if a family's `update()` doesn't return
exactly as many values as its `names()`.


In [68]:
# =====================================================================
#  HISTORICAL REFERENCE  (one pass over the historical segment)
# =====================================================================
def _acf(x: np.ndarray, nlags: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = x - x.mean()
    n = x.size
    d = float(np.dot(x, x))
    if d <= 0.0:
        return np.zeros(nlags)
    return np.array([float(np.dot(x[k:], x[:n - k])) / d for k in range(1, nlags + 1)])


def _yule_walker(r: np.ndarray, p: int) -> np.ndarray:
    R = np.eye(p)
    for i in range(p):
        for j in range(p):
            if i != j:
                R[i, j] = r[abs(i - j) - 1]
    try:
        a = np.linalg.solve(R, r[:p])
    except np.linalg.LinAlgError:
        a = np.zeros(p)
    if not np.all(np.isfinite(a)):
        a = np.zeros(p)
    return a


def _pit_z_of(values: np.ndarray) -> np.ndarray:
    """Map an array to its own PIT-normal scores (used for historical diagnostics)."""
    n = values.size
    order = np.argsort(values, kind="mergesort")
    ranks = np.empty(n, dtype=np.float64)
    ranks[order] = np.arange(n, dtype=np.float64)
    return ndtri((ranks + 0.5) / n)


def _l_moments(sorted_x: np.ndarray):
    """Hosking (1990) unbiased sample L-moments l1..l4 from a SORTED array."""
    n = sorted_x.size
    if n < 4:
        return (0.0, 1e-6, 0.0, 0.0)
    x = sorted_x
    i = np.arange(1, n + 1, dtype=np.float64)
    b0 = x.mean()
    b1 = np.sum((i - 1) / (n - 1) * x) / n
    b2 = np.sum((i - 1) * (i - 2) / ((n - 1) * (n - 2)) * x) / n if n > 2 else 0.0
    b3 = np.sum((i - 1) * (i - 2) * (i - 3) / ((n - 1) * (n - 2) * (n - 3)) * x) / n if n > 3 else 0.0
    l1 = b0
    l2 = 2 * b1 - b0
    l3 = 6 * b2 - 6 * b1 + b0
    l4 = 20 * b3 - 30 * b2 + 12 * b1 - b0
    return (float(l1), float(max(l2, 1e-12)), float(l3), float(l4))


class HistoricalRef:
    """Everything derived once from the (break-free) historical segment."""

    def __init__(self, x_hist):
        xh = np.asarray(x_hist, dtype=np.float64).ravel()
        xh = xh[np.isfinite(xh)]
        if xh.size < 4 * AR_P + 8:                       # pathological guard
            xh = np.concatenate([xh, np.zeros(4 * AR_P + 8 - xh.size)])

        n = xh.size
        self.n = n
        self.mu = float(xh.mean())
        self.sd = max(float(xh.std(ddof=1)), 1e-12)
        med = float(np.median(xh))
        self.mad = max(1.4826 * float(np.median(np.abs(xh - med))), 1e-12)
        self.xmin = float(xh.min())
        self.xmax = float(xh.max())

        # ---- PIT table for the raw value -----------------------------
        self.xs_l = np.sort(xh).tolist()
        self.ztab = ndtri((np.arange(n + 1) + 0.5) / (n + 1)).tolist()

        # ---- standardized moments / tail rates -----------------------
        zr = (xh - self.mu) / self.sd
        self.skew = float((zr ** 3).mean())
        self.kurt = float((zr ** 4).mean() - 3.0)
        self.p2 = max(float((np.abs(zr) > 2.0).mean()), 1.0 / (n + 1))
        self.p3 = max(float((np.abs(zr) > 3.0).mean()), 1.0 / (n + 1))

        # ---- AR(p) whitening ----------------------------------------
        r = _acf(xh, max(AR_P, 2))
        self.acf1, self.acf2 = float(r[0]), float(r[1])
        a = _yule_walker(r, AR_P)
        self.ar_l = [float(v) for v in a]

        c = xh - self.mu
        res = c[AR_P:].copy()
        for k in range(AR_P):
            res -= self.ar_l[k] * c[AR_P - 1 - k: c.size - 1 - k]
        self.rsd = max(float(res.std(ddof=1)), 1e-12)

        m = res.size
        self.es_l = np.sort(res).tolist()
        self.etab = ndtri((np.arange(m + 1) + 0.5) / (m + 1)).tolist()

        # ---- lag seeds so the online stream starts warm ---------------
        zh = _pit_z_of(xh)
        rz = _acf(zh, 2)
        self.zacf1, self.zacf2 = float(rz[0]), float(rz[1])
        self.z_seed1 = float(zh[-1])
        self.z_seed2 = float(zh[-2])
        self.tail_x = [float(v) for v in xh[-AR_P:]]

        eh = _pit_z_of(res)
        self.e_seed1 = float(eh[-1])
        self.e_seed2 = float(eh[-2])

        # ---- context block (constant for the whole series) ------------
        self.context = [
            float(n), math.log(n),
            self.skew, self.kurt, self.acf1, self.acf2,
            self.ar_l[0], self.ar_l[1], self.ar_l[2],
            self.rsd / self.sd, self.sd / self.mad,
            self.p2, self.p3, self.zacf1,
        ]

        # ---- Tier 1-4: deterministic PIT-normal quantile grids, used as the
        # fixed "historical sample" side of every two-sample test (see 1b markdown).
        REF_MAX = 256   # the two-sample reference is a smooth deterministic quantile
                        # grid, not raw data -- 256 knots reproduce it to well within
                        # the sampling noise of any window of length <= 256.
        _zr = np.asarray(self.ztab[1:-1], dtype=np.float64)
        _er = np.asarray(self.etab[1:-1], dtype=np.float64)
        if _zr.size > REF_MAX:
            _zr = np.ascontiguousarray(_zr[np.linspace(0, _zr.size - 1, REF_MAX).astype(int)])
        if _er.size > REF_MAX:
            _er = np.ascontiguousarray(_er[np.linspace(0, _er.size - 1, REF_MAX).astype(int)])
        self.z_ref, self.e_ref = _zr, _er
        self.z_ref_lmom = _l_moments(np.sort(self.z_ref))
        self.e_ref_lmom = _l_moments(np.sort(self.e_ref))


In [ ]:
# =====================================================================
#  CONFIG
# =====================================================================
WINDOWS = (32, 64, 256)      # trailing windows = candidate post-break lengths
TWOSAMPLE_CHANNELS = ("e",)      # was ("z","e"); all 30 top diagnostic features were e-channel
ALPHAS  = (0.02, 0.08, 0.25)     # EWMA decays: slow / medium / fast
DRIFTS  = (0.25, 0.75, 1.50)     # Page-CUSUM reference drifts, in sigma units
ACF_W   = 64                     # window for streaming autocorrelation
SLOPE_W = 64                     # window for streaming regression slope
SIGN_W  = (32, 128)              # windows for the sign / runs statistics
TAIL_W  = 64                     # window for tail-exceedance counts
BIN_W   = 128                    # window for the binned chi-square (shape change)
NBINS   = 8
AR_P    = 3                      # AR order for the whitening channel

CH = ("z", "z2", "e", "e2")      # channels: loc / scale / loc-of-residual / scale-of-residual
LBUF = max(max(WINDOWS), BIN_W, ACF_W, SLOPE_W, max(SIGN_W), TAIL_W) + 1
INV_SQRT2 = 0.7071067811865476

# FEATURES / N_FEATURES are no longer hand-written here -- they're derived from
# REGISTRY once every FeatureFamily has been defined and registered (see the end
# of section 1i, after the Tier 1-4 families). Everything between here and there
# is registry plumbing + the actual family classes.


### 1a. Legacy snapshot -- equivalence baseline ONLY

This is a frozen, renamed copy of the *old* `feature_names()` / `HistoricalRef` /
`StreamFeatures` (identical arithmetic to the previous version of this notebook, before the
registry refactor). Its only job is to be the ground truth for the equivalence check in
section 1e: run both extractors over the same series, assert bit-identical output. Once you've
run that check successfully, this cell is safe to delete -- it plays no role in `train()` /
`infer()` / anything downstream.


### 1c. Registry infrastructure

`ChannelState` is the shared incremental tracker for one channel (ring buffer, per-`WINDOWS`
running sums, exp sum, per-`ALPHAS` EWMA, per-`DRIFTS` CUSUM) -- pushed once per step by the
driver, read by any family that needs this channel (the old code recomputed / re-read this ad
hoc in several places; now it's pushed once, read many times).

`Ctx` is what the driver hands each family's `update()`: this step's `t`/`n`, the 4 channel
values, `zr`/`u` (needed by the tail/chi2 families), the `channels` dict, and `HistoricalRef`.
`Ctx.sorted_window(channel, W)` lazily sorts a trailing window and **caches it for the rest of
this step** -- several Tier 1-4 families want the same (channel, W) sorted view in a single
step, so this avoids re-sorting it once per family.

`stride_for(cost, t)` is a pure function of `t` (never of series content), mirroring
`PREDICT_STRIDE`/`_stride_for` in `infer()` -- the same argument applies: every series is
equally stale at a given `t`, so within-`t` ranking (all TS-AUC cares about) is unaffected.
`"O1"` families run every step (unchanged from before); `"sorted"`/`"heavy"` families -- the
new Tier 1-4 families that need a trailing window slice or a sorted view -- recompute on a
coarser, dense-early/sparse-late schedule and hold their previous value between recomputes.
This is also what bounds the cost of `Ctx.sorted_window`: since it's only reached when a strided
family actually recomputes, the window only gets re-sorted at that same reduced cadence, not
every step -- a full incrementally-maintained sorted structure (`bisect.insort`/evict) would
shave further, and is a reasonable next step if the stride schedule above turns out to be too
coarse, but wasn't necessary to hit the O(W log W)-at-a-stride budget this refactor targets.


In [69]:
CH_IDX = {name: i for i, name in enumerate(CH)}  # z:0 z2:1 e:2 e2:3


class ChannelState:
    """Ring buffer + exp/EWMA/window/CUSUM incremental trackers for one channel.
    Owned by the driver, pushed once per step; any family needing this channel
    reads it off `ctx` instead of recomputing it (e.g. ContrastFamily/SlopeFamily
    both need the window running-sums `rs` that the channel bank also emits from)."""

    def __init__(self):
        L = LBUF
        self.buf = [0.0] * L
        self.rs = [0.0] * len(WINDOWS)
        self.exp_s = 0.0
        self.ew_m = [0.0] * len(ALPHAS)
        self.ew_w2 = [0.0] * len(ALPHAS)
        self.gp = [0.0] * len(DRIFTS)
        self.gn = [0.0] * len(DRIFTS)

    def push(self, v: float, t: int) -> None:
        L = LBUF
        i = t % L
        b = self.buf
        rs = self.rs
        for k in range(len(WINDOWS)):
            rs[k] += v - b[(i - WINDOWS[k]) % L]
        b[i] = v
        self.exp_s += v
        m, w2 = self.ew_m, self.ew_w2
        for k in range(len(ALPHAS)):
            a = ALPHAS[k]
            m[k] = (1.0 - a) * m[k] + a * v
            w2[k] = (1.0 - a) * (1.0 - a) * w2[k] + a * a
        gp, gn = self.gp, self.gn
        for k in range(len(DRIFTS)):
            d = DRIFTS[k]
            p = gp[k] + v - d
            gp[k] = p if p > 0.0 else 0.0
            q = gn[k] - v - d
            gn[k] = q if q > 0.0 else 0.0

    def window(self, W: int, t: int) -> np.ndarray:
        """Last min(W, t+1) raw pushed values, chronological order."""
        n = t + 1
        m = W if n > W else n
        L = LBUF
        i = t % L
        idx = [(i - m + 1 + k) % L for k in range(m)]
        return np.asarray([self.buf[k] for k in idx], dtype=np.float64)


class Ctx:
    __slots__ = ("t", "n", "vals", "zr", "u", "channels", "h", "_sort_cache")

    def __init__(self, t, n, vals, zr, u, channels, h):
        self.t = t
        self.n = n
        self.vals = vals              # dict: "z"->float, "z2"->..., "e"->..., "e2"->...
        self.zr = zr
        self.u = u
        self.channels = channels      # dict: "z"->ChannelState, ...
        self.h = h
        self._sort_cache = {}

    def sorted_window(self, channel: str, W: int) -> np.ndarray:
        key = (channel, W)
        c = self._sort_cache.get(key)
        if c is None:
            w = self.channels[channel].window(W, self.t)
            c = self._sort_cache[key] = np.sort(w)
        return c


def stride_for(cost: str, t: int) -> int:
    """Pure function of t only -- every series at step t is equally stale, so
    within-t ranking is unaffected. Same argument as PREDICT_STRIDE / _stride_for
    in infer(): free speed, no data leakage, no series-dependent staleness."""
    if cost == "O1":
        return 1
    if cost == "sorted":
        if t < 128:
            return 1
        if t < 512:
            return 4
        return 16
    # "heavy"
    if t < 128:
        return 2
    if t < 512:
        return 8
    return 32


def _safe(v: float) -> float:
    return v if math.isfinite(v) else 0.0


def _safe_ratio(num: float, den: float) -> float:
    if den <= 0.0 or not math.isfinite(num) or not math.isfinite(den):
        return 0.0
    v = num / den
    return v if math.isfinite(v) else 0.0


class FeatureFamily:
    """Base class for one entry in REGISTRY. init() does one-off O(n_H) work on
    the historical segment; update() consumes ctx (the shared per-step state) and
    returns len(self.names()) floats, in emit order.

    IMPORTANT: REGISTRY holds one prototype *instance* per family (so a family
    with constructor args, like ChannelBankFamily(channel), only needs to be
    built once). init()/update() mutate `self` (e.g. AutocorrFamily's z1/z2,
    SlopeFamily's slope_B), so every StreamFeatures MUST work on its own
    `clone()` of each family, never the REGISTRY prototype directly --
    build_dataset() runs StreamFeatures for different series concurrently
    (joblib threading backend), and sharing mutable family state across those
    threads would silently corrupt features across series."""
    name: str = "base"
    channels: tuple = ("z", "e")
    cost: str = "O1"

    def names(self) -> List[str]:
        raise NotImplementedError

    def init(self, hist: HistoricalRef) -> None:
        pass

    def update(self, ctx: Ctx) -> List[float]:
        raise NotImplementedError

    def clone(self) -> "FeatureFamily":
        return copy.deepcopy(self)


REGISTRY: List[FeatureFamily] = []


### 1d. Porting the existing bank into families (behavior-preserving)

Same arithmetic as the legacy code in 1a, relocated into `FeatureFamily` classes. Verify this
with the equivalence check in 1e **before** trusting anything past this point.

| Family | Reads from `ctx` | Emits |
|---|---|---|
| `ChannelBankFamily(channel)` x4 | `channels[c]` | exp / EWMA x3 / window x4 / CUSUM x3 (11) |
| `ContrastFamily` | `channels["z"/"e"].rs` | `z_contrast`, `e_contrast` |
| `AutocorrFamily` | `vals["z"]`, own `pbuf/ps/pexp` | `z_acf{1,2}_w64`, `z_acf{1,2}_exp` |
| `SignRunsFamily` | `vals["z"]` | `sign_w32`, `sign_w128`, `sign_exp`, `run_len`, `run_max` |
| `TailFamily` | `zr` | `tail{2,3}_w64`, `tail{2,3}_exp`, `beyond_range_exp`, `max_absz_exp` |
| `Chi2Family` | `u` | `chi2_w128`, `chi2_exp` |
| `SlopeFamily` | `channels["z"].buf`/`.rs` | `slope_w64` |
| `TimeFamily` | `t` | `t_step`, `t_log` -- **always kept**, exempt from every prune rule |
| `ContextFamily` | `h.context` (constant) | the 14 `h_*` features |


In [70]:
class ChannelBankFamily(FeatureFamily):
    """exp / EWMA(3) / window(4) / CUSUM(3) bank for one channel."""

    def __init__(self, channel: str):
        self.channel = channel
        self.name = channel
        self.channels = (channel,)

    def names(self):
        p = self.channel
        n = [p + "_exp"]
        n += ["%s_ew%g" % (p, a) for a in ALPHAS]
        n += ["%s_w%d" % (p, W) for W in WINDOWS]
        n += ["%s_cusum%g" % (p, d) for d in DRIFTS]
        return n

    def update(self, ctx: Ctx):
        cs = ctx.channels[self.channel]
        n = ctx.n
        out = [cs.exp_s / math.sqrt(n)]
        for k in range(len(ALPHAS)):
            out.append(cs.ew_m[k] / math.sqrt(cs.ew_w2[k] + 1e-18))
        for k in range(len(WINDOWS)):
            nw = WINDOWS[k] if n > WINDOWS[k] else n
            out.append(cs.rs[k] / math.sqrt(nw))
        for k in range(len(DRIFTS)):
            out.append(cs.gp[k] if cs.gp[k] > cs.gn[k] else cs.gn[k])
        return out


class ContrastFamily(FeatureFamily):
    name = "contrast"
    channels = ("z", "e")

    def names(self):
        return ["z_contrast", "e_contrast"]

    def update(self, ctx: Ctx):
        out = []
        n = ctx.n
        for ch in ("z", "e"):
            rs = ctx.channels[ch].rs
            n1 = WINDOWS[0] if n > WINDOWS[0] else n
            n2 = (WINDOWS[1] if n > WINDOWS[1] else n) - n1
            if n2 <= 0:
                out.append(0.0)
            else:
                d = rs[0] / n1 - (rs[1] - rs[0]) / n2
                out.append(d * math.sqrt(n1 * n2 / float(n1 + n2)))
        return out


class AutocorrFamily(FeatureFamily):
    name = "acf"
    channels = ("z",)

    def init(self, hist: HistoricalRef):
        self.z1, self.z2 = hist.z_seed1, hist.z_seed2
        L = LBUF
        self.pbuf = [[0.0] * L, [0.0] * L]
        self.ps = [0.0, 0.0]
        self.pexp = [0.0, 0.0]
        self.rho = (hist.zacf1, hist.zacf2)

    def names(self):
        out = []
        for lag in (1, 2):
            out += ["z_acf%d_w%d" % (lag, ACF_W), "z_acf%d_exp" % lag]
        return out

    def update(self, ctx: Ctx):
        z = ctx.vals["z"]
        t, n = ctx.t, ctx.n
        L = LBUF
        i = t % L
        prods = (z * self.z1, z * self.z2)
        self.z2 = self.z1
        self.z1 = z
        out = []
        for k in (0, 1):
            pb = self.pbuf[k]
            self.ps[k] += prods[k] - pb[(i - ACF_W) % L]
            pb[i] = prods[k]
            self.pexp[k] += prods[k]
            nw = ACF_W if n > ACF_W else n
            out.append((self.ps[k] / nw - self.rho[k]) * math.sqrt(nw))
            out.append((self.pexp[k] / n - self.rho[k]) * math.sqrt(n))
        return out


class SignRunsFamily(FeatureFamily):
    name = "sign"
    channels = ("z",)

    def init(self, hist: HistoricalRef):
        L = LBUF
        self.sbuf = [0.0] * L
        self.ssum = [0.0] * len(SIGN_W)
        self.sexp = 0.0
        self.run_len = 0.0
        self.run_max = 0.0
        self.last_sign = 0

    def names(self):
        out = ["sign_w%d" % W for W in SIGN_W]
        out += ["sign_exp", "run_len", "run_max"]
        return out

    def update(self, ctx: Ctx):
        z = ctx.vals["z"]
        t, n = ctx.t, ctx.n
        L = LBUF
        i = t % L
        s = 1.0 if z > 0.0 else 0.0
        for k in range(len(SIGN_W)):
            self.ssum[k] += s - self.sbuf[(i - SIGN_W[k]) % L]
        self.sbuf[i] = s
        self.sexp += s
        out = []
        for k in range(len(SIGN_W)):
            nw = SIGN_W[k] if n > SIGN_W[k] else n
            out.append((self.ssum[k] - 0.5 * nw) / math.sqrt(0.25 * nw))
        out.append((self.sexp - 0.5 * n) / math.sqrt(0.25 * n))
        sg = 1 if z > 0.0 else -1
        if sg == self.last_sign:
            self.run_len += 1.0
        else:
            self.run_len = 1.0
            self.last_sign = sg
        if self.run_len > self.run_max:
            self.run_max = self.run_len
        out.append(self.run_len)
        out.append(self.run_max)
        return out


class TailFamily(FeatureFamily):
    name = "tail"
    channels = ("z",)

    def init(self, hist: HistoricalRef):
        L = LBUF
        self.t2buf = [0.0] * L
        self.t3buf = [0.0] * L
        self.t2w = self.t3w = self.t2e = self.t3e = 0.0
        self.beyond = 0.0
        self.max_absz = 0.0
        self.p2, self.p3 = hist.p2, hist.p3
        self.xmin, self.xmax, self.hn = hist.xmin, hist.xmax, hist.n
        self.mu, self.sd = hist.mu, hist.sd

    def names(self):
        out = []
        for thr in (2, 3):
            out += ["tail%d_w%d" % (thr, TAIL_W), "tail%d_exp" % thr]
        out += ["beyond_range_exp", "max_absz_exp"]
        return out

    def update(self, ctx: Ctx):
        zr = ctx.zr
        t, n = ctx.t, ctx.n
        L = LBUF
        i = t % L
        az = zr if zr >= 0.0 else -zr
        e2 = 1.0 if az > 2.0 else 0.0
        e3 = 1.0 if az > 3.0 else 0.0
        self.t2w += e2 - self.t2buf[(i - TAIL_W) % L]
        self.t3w += e3 - self.t3buf[(i - TAIL_W) % L]
        self.t2buf[i] = e2
        self.t3buf[i] = e3
        self.t2e += e2
        self.t3e += e3
        nw = TAIL_W if n > TAIL_W else n
        out = []
        for cnt_w, cnt_e, p in ((self.t2w, self.t2e, self.p2), (self.t3w, self.t3e, self.p3)):
            out.append((cnt_w - nw * p) / math.sqrt(nw * p * (1.0 - p) + 1e-12))
            out.append((cnt_e - n * p) / math.sqrt(n * p * (1.0 - p) + 1e-12))
        xv = self.mu + zr * self.sd
        if xv < self.xmin or xv > self.xmax:
            self.beyond += 1.0
        pb_rate = 2.0 / (self.hn + 1.0)
        out.append((self.beyond - n * pb_rate) / math.sqrt(n * pb_rate * (1.0 - pb_rate) + 1e-12))
        if az > self.max_absz:
            self.max_absz = az
        out.append(self.max_absz)
        return out


class Chi2Family(FeatureFamily):
    name = "chi2"
    channels = ()

    def init(self, hist: HistoricalRef):
        L = LBUF
        self.bbuf = [0.0] * L
        self.cnt_w = [0.0] * NBINS
        self.cnt_e = [0.0] * NBINS

    def names(self):
        return ["chi2_w%d" % BIN_W, "chi2_exp"]

    def update(self, ctx: Ctx):
        u, t, n = ctx.u, ctx.t, ctx.n
        L = LBUF
        i = t % L
        bi = int(u * NBINS)
        if bi >= NBINS:
            bi = NBINS - 1
        if t >= BIN_W:
            self.cnt_w[int(self.bbuf[(i - BIN_W) % L])] -= 1.0
        self.bbuf[i] = float(bi)
        self.cnt_w[bi] += 1.0
        self.cnt_e[bi] += 1.0
        nw = BIN_W if n > BIN_W else n
        ew = nw / NBINS
        chi = sum((self.cnt_w[k] - ew) ** 2 for k in range(NBINS)) / ew
        v1 = (chi - (NBINS - 1)) / math.sqrt(2.0 * (NBINS - 1))
        ee = n / NBINS
        chi = sum((self.cnt_e[k] - ee) ** 2 for k in range(NBINS)) / ee
        v2 = (chi - (NBINS - 1)) / math.sqrt(2.0 * (NBINS - 1))
        return [v1, v2]


class SlopeFamily(FeatureFamily):
    name = "slope"
    channels = ("z",)

    def init(self, hist: HistoricalRef):
        self.slope_B = 0.0

    def names(self):
        return ["slope_w%d" % SLOPE_W]

    def update(self, ctx: Ctx):
        z = ctx.vals["z"]
        t, n = ctx.t, ctx.n
        L = LBUF
        i = t % L
        cs = ctx.channels["z"]
        zb = cs.buf
        self.slope_B += t * z - (t - SLOPE_W) * zb[(i - SLOPE_W) % L]
        nw = SLOPE_W if n > SLOPE_W else n
        A = cs.rs[WINDOWS.index(SLOPE_W)] if SLOPE_W in WINDOWS else 0.0
        jbar = t - (nw - 1) / 2.0
        sxx = nw * (nw * nw - 1.0) / 12.0
        return [(self.slope_B - jbar * A) / math.sqrt(sxx + 1e-12)]


class TimeFamily(FeatureFamily):
    """Pure functions of t. Structurally 0.5 TS-AUC alone -- kept always, exempt
    from every prune/ablate rule (only useful through interactions)."""
    name = "time"
    channels = ()

    def names(self):
        return ["t_step", "t_log"]

    def update(self, ctx: Ctx):
        return [float(ctx.t), math.log1p(ctx.t)]


class ContextFamily(FeatureFamily):
    """Constant per series -- carries no break evidence by itself (design
    principle #5); lets the model learn how much to trust other families for
    this kind of process. Judged by ablation, not univariate AUC."""
    name = "h"
    channels = ()

    def init(self, hist: HistoricalRef):
        self.context = hist.context

    def names(self):
        return ["h_n", "h_logn", "h_skew", "h_kurt", "h_acf1", "h_acf2",
                 "h_ar1", "h_ar2", "h_ar3", "h_rsd_ratio", "h_sd_mad",
                 "h_p2", "h_p3", "h_zacf1"]

    def update(self, ctx: Ctx):
        return list(self.context)


for _ch in CH:
    REGISTRY.append(ChannelBankFamily(_ch))
REGISTRY.append(ContrastFamily())
REGISTRY.append(AutocorrFamily())
REGISTRY.append(SignRunsFamily())
REGISTRY.append(TailFamily())
REGISTRY.append(Chi2Family())
REGISTRY.append(SlopeFamily())
REGISTRY.append(TimeFamily())
REGISTRY.append(ContextFamily())

print("ported families registered:", len(REGISTRY),
      "| features so far:", sum(len(f.names()) for f in REGISTRY))


ported families registered: 12 | features so far: 80


In [71]:
# =====================================================================
#  STREAMING FEATURE EXTRACTOR  (thin registry driver)
# =====================================================================
class StreamFeatures:
    """Same per-step cost model as before (O(1) for the ported bank; strided
    O(W) / O(W log W) for the new Tier 1-4 families). `active_families=None`
    (default) runs every family in REGISTRY; pass a set of family `.name`s to
    restrict to a subset -- this is how `infer()` and the crunch-submission
    `train()` stay fast (compute ACTIVE_FAMILIES only), while the offline
    iteration loop always extracts with active_families=None (full registry).

    Each family is `.clone()`-d from its REGISTRY prototype (see the note on
    FeatureFamily.clone()) so this instance's per-series state is fully its
    own -- required for build_dataset()'s threaded parallelism to be safe."""

    def __init__(self, x_hist, active_families=None):
        h = self.h = HistoricalRef(x_hist)
        self.families = [f.clone() for f in REGISTRY
                          if active_families is None or f.name in active_families]
        for f in self.families:
            f.init(h)
        self.names = [nm for f in self.families for nm in f.names()]
        self.channels = {name: ChannelState() for name in CH}
        self.tx = list(h.tail_x)
        self.t = -1

    def update(self, x) -> List[float]:
        h = self.h
        self.t += 1
        t = self.t
        n = t + 1

        xv = float(x)
        if xv != xv or xv in (float("inf"), float("-inf")):
            xv = h.mu

        # ---- channel 0/1 : PIT of the raw value ----------------------
        j = bisect.bisect_left(h.xs_l, xv)
        z = h.ztab[j]
        u = (j + 0.5) / (h.n + 1.0)
        zr = (xv - h.mu) / h.sd

        # ---- channel 2/3 : PIT of the AR residual --------------------
        tx = self.tx
        rres = xv - h.mu
        for k in range(AR_P):
            rres -= h.ar_l[k] * (tx[AR_P - 1 - k] - h.mu)
        tx.pop(0)
        tx.append(xv)
        je = bisect.bisect_left(h.es_l, rres)
        ze = h.etab[je]

        vals = {"z": z, "z2": (z * z - 1.0) * INV_SQRT2, "e": ze, "e2": (ze * ze - 1.0) * INV_SQRT2}
        for name, v in vals.items():
            self.channels[name].push(v, t)

        ctx = Ctx(t, n, vals, zr, u, self.channels, h)

        out = []
        for f in self.families:
            stride = stride_for(f.cost, t)
            if stride == 1 or t % stride == 0 or not hasattr(f, "_last"):
                r = f.update(ctx)
                assert len(r) == len(f.names()), (
                    "%s.update() returned %d values, names() has %d"
                    % (f.name, len(r), len(f.names())))
                f._last = r
            else:
                r = f._last
            out.extend(r)
        return out


### 1e. Equivalence check -- RUN THIS FIRST, before anything past this point

Runs the legacy extractor (1a) and the registry-based one (1d) over the same ~50 series and
asserts bit-identical output, column for column, in the same order. This is the gate: don't
trust the diagnostic, the cache, or `train()`/`infer()` until this passes. If it fails, the
port has a bug -- go fix it before adding a single Tier 1-4 feature.


In [72]:
def _run_equivalence_check(datasets, n_series=50, atol=1e-6, rtol=1e-6):
    ported_names = [nm for f in REGISTRY if f.name in FEATURES_LEGACY_FAMILY_NAMES
                    for nm in f.names()]
    assert ported_names == FEATURES_LEGACY, (
        "name/order mismatch between the ported registry and the legacy extractor -- "
        "this must be fixed before trusting the port.")

    items = list(datasets)[:n_series]
    max_abs_diff = 0.0
    for did, xh, xo, tau in items:
        fl = StreamFeaturesLegacy(xh)
        fn = StreamFeatures(xh, active_families=FEATURES_LEGACY_FAMILY_NAMES)
        for i, point in enumerate(xo):
            rl = np.asarray(fl.update(point), dtype=np.float64)
            rn = np.asarray(fn.update(point), dtype=np.float64)
            d = float(np.max(np.abs(rl - rn)))
            max_abs_diff = max(max_abs_diff, d)
            if not np.allclose(rl, rn, atol=atol, rtol=rtol):
                bad = int(np.argmax(np.abs(rl - rn)))
                raise AssertionError(
                    "series %s step %d feature %s: legacy=%.10g new=%.10g"
                    % (did, i, ported_names[bad], rl[bad], rn[bad]))
    print("EQUIVALENCE OK over %d series, max abs diff = %.3e" % (len(items), max_abs_diff))


# The ported families' `.name`s -- everything in REGISTRY that existed before this refactor.
FEATURES_LEGACY_FAMILY_NAMES = {f.name for f in REGISTRY}

_run_equivalence_check(train_data, n_series=50)


EQUIVALENCE OK over 50 series, max abs diff = 0.000e+00


### 1f. Tier 1 -- the moment ladder (new features start here)

Every family below is a **two-sample test: historical reference (`h.z_ref`/`e_ref`) vs. the
trailing window ending at `t`**, evaluated at every length in `WINDOWS`, on both the `z` and
`e` channels (so `t_welch_z_w16`, `t_welch_z_w32`, ..., `t_welch_e_w256`, etc. -- 8 features per
family). Where a closed-form null exists we emit the standardized statistic directly (that's
what makes it pivotal): Welch t, Mann-Whitney (rank-sum, normal approx), F-test / Bartlett /
Levene / Fligner-Killeen (scale), D'Agostino skewness and Anscombe-Glynn kurtosis (both have
known `N(0, c/n)` sampling variance), Jarque-Bera and D'Agostino K² (both `~chi2(2)`, standardized
like the existing `chi2_*` features). Location (Welch t, Mann-Whitney) is expected to be weak
given the diagnostic -- implemented anyway, cheap, and it establishes the noise floor.

`TwoSampleFamily` is the shared base: it derives `names()` from `(stat, channel, W)`, pulls the
window (raw or sorted, per subclass) via `Ctx`, and standardizes with a guard against
degenerate windows (returns 0.0, never NaN/inf).


In [ ]:
def _mean_var(x: np.ndarray):
    n = x.size
    if n < 2:
        return 0.0, 1e-6
    mu = float(x.mean())
    v = float(x.var(ddof=1))
    return mu, max(v, 1e-12)


def _avg_ranks(x: np.ndarray) -> np.ndarray:
    n = x.size
    if n == 0:
        return np.empty(0, dtype=np.float64)
    order = np.argsort(x, kind="mergesort")
    s = x[order]
    neq = np.empty(n, dtype=bool)
    neq[0] = True
    if n > 1:
        np.not_equal(s[1:], s[:-1], out=neq[1:])
    starts = np.flatnonzero(neq)
    grp = np.cumsum(neq) - 1
    counts = np.diff(np.append(starts, n))
    avg = starts.astype(np.float64) + (counts.astype(np.float64) + 1.0) / 2.0
    out = np.empty(n, dtype=np.float64)
    out[order] = avg[grp]
    return out


def _guard(m: int, nr: int, min_m: int = 4) -> bool:
    return m >= min_m and nr >= min_m


class RefSub:
    """Fixed-size random subsample of a reference array, used by O(m*n_ref)
    pairwise-cost families (Tier 3 energy distance / MMD) so a large n_H
    doesn't blow up the per-step cost."""
    CAP = 192

    def __init__(self, ref: np.ndarray, seed: int = 0):
        if ref.size <= self.CAP:
            self.arr = ref
        else:
            rng = np.random.default_rng(seed)
            self.arr = rng.choice(ref, size=self.CAP, replace=False)


class TwoSampleFamily(FeatureFamily):
    cost = "sorted"
    needs_sorted = True
    channels = TWOSAMPLE_CHANNELS      # <-- add this line
    def names(self):
        return ["%s_%s_w%d" % (self.name, ch, W) for ch in self.channels for W in WINDOWS]

    def init(self, hist: HistoricalRef):
        self.ref = {"z": hist.z_ref, "e": hist.e_ref}

    def _window(self, ctx: Ctx, ch: str, W: int) -> np.ndarray:
        if self.needs_sorted:
            return ctx.sorted_window(ch, W)
        return ctx.channels[ch].window(W, ctx.t)

    def stat(self, w: np.ndarray, r: np.ndarray) -> float:
        raise NotImplementedError

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            ref = self.ref[ch]
            for W in WINDOWS:
                w = self._window(ctx, ch, W)
                out.append(_safe(self.stat(w, ref)))
        return out


class WelchTFamily(TwoSampleFamily):
    name = "welch"
    needs_sorted = False

    def stat(self, w, r):
        m, nr = w.size, r.size
        if not _guard(m, nr):
            return 0.0
        m1, v1 = _mean_var(w)
        m2, v2 = _mean_var(r)
        se = math.sqrt(v1 / m + v2 / nr)
        return (m1 - m2) / se if se > 0 else 0.0


class MannWhitneyFamily(TwoSampleFamily):
    name = "mwu"

    def stat(self, w, r):
        m, nr = w.size, r.size
        if not _guard(m, nr):
            return 0.0
        combined = np.concatenate([w, r])
        ranks = _avg_ranks(combined)
        rank_sum_w = ranks[:m].sum()
        u = rank_sum_w - m * (m + 1) / 2.0
        mean = m * nr / 2.0
        var = m * nr * (m + nr + 1) / 12.0
        return (u - mean) / math.sqrt(var) if var > 0 else 0.0


class FTestFamily(TwoSampleFamily):
    name = "ftest"
    needs_sorted = False

    def stat(self, w, r):
        m, nr = w.size, r.size
        if not _guard(m, nr):
            return 0.0
        _, v1 = _mean_var(w)
        _, v2 = _mean_var(r)
        logf = math.log(v1 / v2)
        se = math.sqrt(2.0 / (m - 1) + 2.0 / (nr - 1))
        return logf / se if se > 0 else 0.0


class BartlettFamily(TwoSampleFamily):
    name = "bartlett"
    needs_sorted = False

    def stat(self, w, r):
        m, nr = w.size, r.size
        if not _guard(m, nr):
            return 0.0
        _, v1 = _mean_var(w)
        _, v2 = _mean_var(r)
        sp2 = ((m - 1) * v1 + (nr - 1) * v2) / (m + nr - 2)
        b = (m + nr - 2) * math.log(sp2) - (m - 1) * math.log(v1) - (nr - 1) * math.log(v2)
        c = 1.0 + (1.0 / 3.0) * (1.0 / (m - 1) + 1.0 / (nr - 1) - 1.0 / (m + nr - 2))
        chi2_1 = b / c if c > 0 else 0.0
        return (chi2_1 - 1.0) / math.sqrt(2.0)


class LeveneFamily(TwoSampleFamily):
    """Brown-Forsythe variant (median-centered) -- robust to non-normality."""
    name = "levene"
    needs_sorted = False

    def stat(self, w, r):
        m, nr = w.size, r.size
        if not _guard(m, nr):
            return 0.0
        zw = np.abs(w - np.median(w))
        zr = np.abs(r - np.median(r))
        zbar_w, zbar_r = zw.mean(), zr.mean()
        zbar = (zw.sum() + zr.sum()) / (m + nr)
        ssb = m * (zbar_w - zbar) ** 2 + nr * (zbar_r - zbar) ** 2
        ssw = float(((zw - zbar_w) ** 2).sum() + ((zr - zbar_r) ** 2).sum())
        df2 = m + nr - 2
        if ssw <= 0 or df2 <= 0:
            return 0.0
        f = ssb / (ssw / df2)
        sign = 1.0 if zbar_w >= zbar_r else -1.0
        return sign * math.sqrt(max(f, 0.0))


class FlignerKilleenFamily(TwoSampleFamily):
    """Rank-based (normal-scores) scale test -- distribution-free, robust to tails."""
    name = "fligner"

    def stat(self, w, r):
        m, nr = w.size, r.size
        if not _guard(m, nr):
            return 0.0
        aw = np.abs(w - np.median(w))
        ar = np.abs(r - np.median(r))
        combined = np.concatenate([aw, ar])
        n = combined.size
        ranks = _avg_ranks(combined)
        scores = ndtri(0.5 + ranks / (2.0 * (n + 1)))
        sum_w = scores[:m].sum()
        mean_score = scores.mean()
        var_score = scores.var(ddof=1) if n > 1 else 1e-6
        mean = m * mean_score
        var = m * nr / n * var_score if n > 0 else 1e-6
        return (sum_w - mean) / math.sqrt(max(var, 1e-9))


class SkewFamily(TwoSampleFamily):
    """D'Agostino skewness g1, standardized ~N(0, 6/n)."""
    name = "skew"
    needs_sorted = False

    def stat(self, w, r):
        m = w.size
        if m < 8:
            return 0.0
        mu, v = _mean_var(w)
        s = math.sqrt(v)
        if s <= 0:
            return 0.0
        g1 = float(((w - mu) ** 3).mean()) / s ** 3
        return g1 / math.sqrt(6.0 / m)


class KurtFamily(TwoSampleFamily):
    """Anscombe-Glynn excess kurtosis g2, standardized ~N(0, 24/n)."""
    name = "kurt"
    needs_sorted = False

    def stat(self, w, r):
        m = w.size
        if m < 8:
            return 0.0
        mu, v = _mean_var(w)
        s2 = v
        if s2 <= 0:
            return 0.0
        g2 = float(((w - mu) ** 4).mean()) / s2 ** 2 - 3.0
        return g2 / math.sqrt(24.0 / m)


class JarqueBeraFamily(TwoSampleFamily):
    """JB ~ chi2(2), standardized like the existing chi2_* features."""
    name = "jb"
    needs_sorted = False

    def stat(self, w, r):
        m = w.size
        if m < 8:
            return 0.0
        mu, v = _mean_var(w)
        s = math.sqrt(v)
        if s <= 0:
            return 0.0
        g1 = float(((w - mu) ** 3).mean()) / s ** 3
        g2 = float(((w - mu) ** 4).mean()) / s ** 4 - 3.0
        jb = m / 6.0 * (g1 ** 2 + g2 ** 2 / 4.0)
        return (jb - 2.0) / 2.0


class DAgostinoK2Family(TwoSampleFamily):
    """D'Agostino-Pearson omnibus K^2 ~ chi2(2) -- the scipy.stats.normaltest
    algorithm, reimplemented directly so the online path stays dependency-free."""
    name = "dak2"
    needs_sorted = False

    def stat(self, w, r):
        n = w.size
        if n < 20:
            return 0.0
        mu, v = _mean_var(w)
        s = math.sqrt(v)
        if s <= 0:
            return 0.0
        g1 = float(((w - mu) ** 3).mean()) / s ** 3
        g2 = float(((w - mu) ** 4).mean()) / s ** 4 - 3.0

        y = g1 * math.sqrt((n + 1) * (n + 3) / (6.0 * (n - 2)))
        beta2 = (3.0 * (n ** 2 + 27 * n - 70) * (n + 1) * (n + 3) /
                 ((n - 2) * (n + 5) * (n + 7) * (n + 9)))
        w2m1 = 2 * (beta2 - 1.0)
        if w2m1 <= 0:
            return 0.0
        w2 = -1.0 + math.sqrt(w2m1)
        if w2 <= 1.0:
            return 0.0
        delta = 1.0 / math.sqrt(0.5 * math.log(w2))
        alpha = math.sqrt(2.0 / (w2 - 1.0))
        arg = y / alpha
        z1 = delta * math.log(arg + math.sqrt(arg * arg + 1.0))

        e = 3.0 * (n - 1) / (n + 1)
        var = 24.0 * n * (n - 2) * (n - 3) / ((n + 1) ** 2 * (n + 3) * (n + 5))
        if var <= 0:
            return 0.0
        x = (g2 - e) / math.sqrt(var)
        sqrtb1 = (6.0 * (n * n - 5 * n + 2) / ((n + 7) * (n + 9)) *
                  math.sqrt(6.0 * (n + 3) * (n + 5) / (n * (n - 2) * (n - 3))))
        if sqrtb1 <= 0:
            return 0.0
        a = 6.0 + 8.0 / sqrtb1 * (2.0 / sqrtb1 + math.sqrt(1.0 + 4.0 / sqrtb1 ** 2))
        term = (1.0 - 2.0 / a) / (1.0 + x * math.sqrt(2.0 / (a - 4.0)))
        if term <= 0 or a <= 0:
            return 0.0
        z2 = ((1.0 - 2.0 / (9.0 * a)) - term ** (1.0 / 3.0)) / math.sqrt(2.0 / (9.0 * a))

        k2 = z1 * z1 + z2 * z2
        return (k2 - 2.0) / 2.0


REGISTRY.extend([
    WelchTFamily(), MannWhitneyFamily(), FTestFamily(), BartlettFamily(),
    LeveneFamily(), FlignerKilleenFamily(), SkewFamily(), KurtFamily(),
    JarqueBeraFamily(), DAgostinoK2Family(),
])
print("Tier 1 registered | features so far:", sum(len(f.names()) for f in REGISTRY))


Tier 1 registered | features so far: 160


### 1g. Tier 2 -- robust analogues (highest priority)

The diagnostic says we currently win on essentially **one** scale statistic (`e2_*`). This tier
gives the model several more, with different robustness/efficiency profiles: L-moments
(L-scale/L-skew/L-kurt -- linear combinations of order statistics, far lower sampling variance
than conventional moments under heavy tails), quantile-based shape (IQR ratio, Bowley
skewness, Moors' octile-based kurtosis), and three rank-based distribution-free scale tests
(Ansari-Bradley, Mood, Siegel-Tukey -- the scale analogue of Mann-Whitney).

Where no simple closed-form standard error exists (L-moments, Bowley, Moors), the
standardization below is a documented, dimensionally-consistent approximation (deviation from
the reference value, scaled by `~1/sqrt(m)`) rather than an exact asymptotic pivot. That's a
deliberate tradeoff, not an oversight: the permutation-null diagnostic in section 3 is exactly
the tool that empirically checks whether each of these clears a real noise floor, so an
imprecise-but-monotone-and-finite standardization is a power question to measure there, not a
correctness risk here.


In [ ]:
class LMomentsFamily(FeatureFamily):
    """L-scale, L-skewness, L-kurtosis vs. the reference grid's L-moments."""
    name = "lmom"
    cost = "sorted"

    def init(self, hist: HistoricalRef):
        self.ref_lmom = {"z": hist.z_ref_lmom, "e": hist.e_ref_lmom}

    def names(self):
        out = []
        for ch in self.channels:
            for stat in ("lscale", "lskew", "lkurt"):
                for W in WINDOWS:
                    out.append("%s_%s_%s_w%d" % (self.name, stat, ch, W))
        return out

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            l1r, l2r, l3r, l4r = self.ref_lmom[ch]
            t3r = l3r / l2r
            t4r = l4r / l2r
            for W in WINDOWS:
                w = ctx.sorted_window(ch, W)
                m = w.size
                if m < 8:
                    out += [0.0, 0.0, 0.0]
                    continue
                l1, l2, l3, l4 = _l_moments(w)
                out.append(_safe(math.log(max(l2, 1e-9) / l2r) * math.sqrt(m - 1)))
                out.append(_safe((l3 / l2 - t3r) * math.sqrt(m)))
                out.append(_safe((l4 / l2 - t4r) * math.sqrt(m)))
        return out


class IQRRatioFamily(TwoSampleFamily):
    name = "iqr"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.ref_iqr = {ch: float(np.subtract(*np.percentile(self.ref[ch], [75, 25])))
                         for ch in ("z", "e")}

    def stat(self, w, r):
        m = w.size
        if m < 8:
            return 0.0
        ch = self._cur_ch
        q75, q25 = np.percentile(w, [75, 25])
        iqr_w = float(q75 - q25)
        iqr_r = self.ref_iqr[ch]
        if iqr_w <= 0 or iqr_r <= 0:
            return 0.0
        return math.log(iqr_w / iqr_r) * math.sqrt(max(m - 1, 1))

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            self._cur_ch = ch
            for W in WINDOWS:
                w = self._window(ctx, ch, W)
                out.append(_safe(self.stat(w, self.ref[ch])))
        return out


class BowleySkewFamily(TwoSampleFamily):
    """(Q3 + Q1 - 2*Q2) / (Q3 - Q1) -- quartile-based skewness."""
    name = "bowley"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.ref_b = {ch: self._bowley(self.ref[ch]) for ch in ("z", "e")}

    @staticmethod
    def _bowley(x):
        if x.size < 4:
            return 0.0
        q1, q2, q3 = np.percentile(x, [25, 50, 75])
        d = q3 - q1
        return float((q3 + q1 - 2 * q2) / d) if d > 0 else 0.0

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            ref_b = self.ref_b[ch]
            for W in WINDOWS:
                w = self._window(ctx, ch, W)
                m = w.size
                if m < 8:
                    out.append(0.0)
                    continue
                out.append(_safe((self._bowley(w) - ref_b) * math.sqrt(m)))
        return out


class MoorsKurtosisFamily(TwoSampleFamily):
    """Octile-based kurtosis: ((E7-E5)+(E3-E1)) / (E6-E2)."""
    name = "moors"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.ref_t = {ch: self._moors(self.ref[ch]) for ch in ("z", "e")}

    @staticmethod
    def _moors(x):
        if x.size < 8:
            return 0.0
        e1, e2, e3, e5, e6, e7 = np.percentile(x, [12.5, 25, 37.5, 62.5, 75, 87.5])
        den = e6 - e2
        return float(((e7 - e5) + (e3 - e1)) / den) if den > 0 else 0.0

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            ref_t = self.ref_t[ch]
            for W in WINDOWS:
                w = self._window(ctx, ch, W)
                m = w.size
                if m < 16:
                    out.append(0.0)
                    continue
                out.append(_safe((self._moors(w) - ref_t) * math.sqrt(m)))
        return out


class AnsariBradleyFamily(TwoSampleFamily):
    """Rank-based scale test: score = min(rank, N+1-rank) (ends-inward ranking)."""
    name = "ansari"

    def stat(self, w, r):
        m, nr = w.size, r.size
        if not _guard(m, nr):
            return 0.0
        n = m + nr
        combined = np.concatenate([w, r])
        ranks = _avg_ranks(combined)
        scores = np.minimum(ranks, n + 1 - ranks)
        ab = scores[:m].sum()
        if n % 2 == 0:
            e = m * (n + 2) / 4.0
            var = m * nr * (n + 2) * (n - 2) / (48.0 * (n - 1))
        else:
            e = m * (n + 1) ** 2 / (4.0 * n)
            var = m * nr * (n + 1) * (3 + n * n) / (48.0 * n * n)
        return (ab - e) / math.sqrt(var) if var > 0 else 0.0


class MoodFamily(TwoSampleFamily):
    """Rank-based scale test: score = (rank - (N+1)/2)^2."""
    name = "mood"

    def stat(self, w, r):
        m, nr = w.size, r.size
        if not _guard(m, nr):
            return 0.0
        n = m + nr
        combined = np.concatenate([w, r])
        ranks = _avg_ranks(combined)
        scores = (ranks - (n + 1) / 2.0) ** 2
        t = scores[:m].sum()
        e = m * (n * n - 1) / 12.0
        var = m * nr * (n + 1) * (n * n - 4) / 180.0
        return (t - e) / math.sqrt(var) if var > 0 else 0.0


class SiegelTukeyFamily(TwoSampleFamily):
    """Ends-alternating ('zig-zag') ranking, then a Wilcoxon rank-sum on it."""
    name = "sigtuk"

    def stat(self, w, r):
        m, nr = w.size, r.size
        if not _guard(m, nr):
            return 0.0
        n = m + nr
        combined = np.concatenate([w, r])
        order = np.argsort(combined, kind="mergesort")
        st_rank = np.empty(n, dtype=np.float64)
        n_lo = (n + 1) // 2
        n_hi = n // 2
        st_rank[order[:n_lo]] = 2.0 * np.arange(n_lo) + 1.0          # 1,3,5,... from the low end
        if n_hi:
            st_rank[order[n - 1:n - 1 - n_hi:-1]] = 2.0 * np.arange(n_hi) + 2.0   # 2,4,6,... from the high end
        rank_sum_w = st_rank[:m].sum()
        mean = m * (n + 1) / 2.0
        var = m * nr * (n + 1) / 12.0
        return (rank_sum_w - mean) / math.sqrt(var) if var > 0 else 0.0


REGISTRY.extend([
    LMomentsFamily(), IQRRatioFamily(), BowleySkewFamily(), MoorsKurtosisFamily(),
    AnsariBradleyFamily(), MoodFamily(), SiegelTukeyFamily(),
])
print("Tier 2 registered | features so far:", sum(len(f.names()) for f in REGISTRY))


Tier 2 registered | features so far: 232


### 1h. Tier 3 -- omnibus / full-distribution distance

Cramér-von Mises (integrated squared EDF distance, more central power than KS), Anderson-Darling
(tail-weighted, complements CvM), Wasserstein-1, Hellinger and Jensen-Shannon divergence
(binned against the reference's own quantile grid, `NB=10` bins), energy distance, and MMD with
an RBF kernel (bandwidth via the median-pairwise-distance heuristic).

**KS is not reimplemented here.** The causal sup-KS scan near the bottom of the notebook
(`ks_scan_scores`) already is this test, fully causal, and is evaluated separately as a
standalone competing baseline against the whole LightGBM bank -- see the markdown above that
cell. Folding it into the registry as a per-step feature is a reasonable follow-up if the 10k
diagnostic shows the standalone baseline outperforming the bank on the KS axis specifically;
left out for now as a deliberate scope call, not a silent gap.

`cost = "heavy"`: the EDF-distance and kernel families are `O(W log W)` to `O(W * n_ref_sub)`
per evaluation (pairwise distances for energy distance / MMD, capped via `RefSub` at 192
reference points so a large `n_H` doesn't blow up the cost) -- these get the coarsest stride.


In [75]:
class CvMFamily(TwoSampleFamily):
    """Two-sample Cramer-von Mises (Anderson 1962 form), standardized against
    the known asymptotic mean/variance of the omega^2 statistic."""
    name = "cvm"
    cost = "heavy"

    def stat(self, w, r):
        m, nr = w.size, r.size
        if not _guard(m, nr, 8):
            return 0.0
        n = m + nr
        combined = np.concatenate([w, r])
        ranks = _avg_ranks(combined)
        rw = np.sort(ranks[:m])
        rr = np.sort(ranks[m:])
        i1 = np.arange(1, m + 1)
        i2 = np.arange(1, nr + 1)
        u = (m * np.sum((rw - i1) ** 2) + nr * np.sum((rr - i2) ** 2)) / (m * nr * n) - \
            (4.0 * m * nr - 1.0) / (6.0 * n)
        return (u - 1.0 / 6.0) / math.sqrt(1.0 / 45.0)


class ADFamily(TwoSampleFamily):
    """Discretized two-sample Anderson-Darling (tail-weighted EDF distance)."""
    name = "ad"
    cost = "heavy"

    def stat(self, w, r):
        m, nr = w.size, r.size
        if not _guard(m, nr, 8):
            return 0.0
        n = m + nr
        combined = np.sort(np.concatenate([w, r]))
        z_sorted = np.sort(w)
        r_sorted = np.sort(r)
        fw = np.searchsorted(z_sorted, combined[:-1], side="right") / m
        fr = np.searchsorted(r_sorted, combined[:-1], side="right") / nr
        i = np.arange(1, n)
        frac = i / n
        weight = frac * (1.0 - frac)
        weight = np.where(weight > 1e-6, weight, 1e-6)
        a2 = float(np.sum((fw - fr) ** 2 / weight)) * (m * nr) / (n * n)
        return (a2 - 1.0) * math.sqrt(max(m, 1))


class WassersteinFamily(TwoSampleFamily):
    name = "wass"
    cost = "heavy"

    def stat(self, w, r):
        m, nr = w.size, r.size
        if not _guard(m, nr, 8):
            return 0.0
        q = (np.arange(1, m + 1) - 0.5) / m
        rq = np.quantile(r, q)
        w1 = float(np.abs(w - rq).mean())
        return w1 * math.sqrt(m)


class HellingerFamily(TwoSampleFamily):
    """Binned Hellinger distance against the reference's own decile grid."""
    name = "hell"
    cost = "heavy"
    NB = 10

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.edges = {ch: np.quantile(self.ref[ch], np.linspace(0, 1, self.NB + 1))
                      for ch in ("z", "e")}

    def _hist(self, x, edges):
        edges = edges.copy()
        edges[0], edges[-1] = -np.inf, np.inf
        c, _ = np.histogram(x, bins=edges)
        p = c.astype(np.float64) + 0.5
        return p / p.sum()

    def stat(self, w, r):
        if w.size < 8:
            return 0.0
        ch = self._cur_ch
        edges = self.edges[ch]
        pw = self._hist(w, edges)
        pr = self._hist(r, edges)
        h = math.sqrt(0.5 * float(np.sum((np.sqrt(pw) - np.sqrt(pr)) ** 2)))
        return h * math.sqrt(w.size)

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            self._cur_ch = ch
            for W in WINDOWS:
                w = self._window(ctx, ch, W)
                out.append(_safe(self.stat(w, self.ref[ch])))
        return out


class JSFamily(HellingerFamily):
    """Binned Jensen-Shannon divergence, same binning as HellingerFamily."""
    name = "js"

    def stat(self, w, r):
        if w.size < 8:
            return 0.0
        ch = self._cur_ch
        edges = self.edges[ch]
        pw = self._hist(w, edges)
        pr = self._hist(r, edges)
        mmix = 0.5 * (pw + pr)
        kl_w = float(np.sum(pw * np.log(pw / mmix)))
        kl_r = float(np.sum(pr * np.log(pr / mmix)))
        js = 0.5 * kl_w + 0.5 * kl_r
        return js * w.size


class EnergyDistanceFamily(TwoSampleFamily):
    name = "energy"
    cost = "heavy"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.refsub = {ch: RefSub(hist.z_ref if ch == "z" else hist.e_ref).arr for ch in ("z", "e")}

    def stat(self, w, r):
        m = w.size
        if m < 8:
            return 0.0
        rs = r
        wr = np.abs(w[:, None] - rs[None, :]).mean()
        ww = np.abs(w[:, None] - w[None, :]).mean()
        rr = np.abs(rs[:, None] - rs[None, :]).mean()
        e = 2 * wr - ww - rr
        return max(e, 0.0) * math.sqrt(m)

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            for W in WINDOWS:
                w = self._window(ctx, ch, W)
                out.append(_safe(self.stat(w, self.refsub[ch])))
        return out


class MMDRbfFamily(EnergyDistanceFamily):
    name = "mmd"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        allref = np.concatenate([self.refsub["z"], self.refsub["e"]])
        d = np.abs(allref[:, None] - allref[None, :])
        med = np.median(d[d > 0]) if np.any(d > 0) else 1.0
        self.sigma2 = max(float(med) ** 2, 1e-6)

    def stat(self, w, r):
        m = w.size
        if m < 8:
            return 0.0
        rs = r
        s2 = self.sigma2
        kww = np.exp(-(w[:, None] - w[None, :]) ** 2 / (2 * s2)).mean()
        krr = np.exp(-(rs[:, None] - rs[None, :]) ** 2 / (2 * s2)).mean()
        kwr = np.exp(-(w[:, None] - rs[None, :]) ** 2 / (2 * s2)).mean()
        mmd2 = kww + krr - 2 * kwr
        return max(mmd2, 0.0) * m


REGISTRY.extend([
    CvMFamily(), ADFamily(), WassersteinFamily(), HellingerFamily(), JSFamily(),
    EnergyDistanceFamily(), MMDRbfFamily(),
])
print("Tier 3 registered | features so far:", sum(len(f.names()) for f in REGISTRY))


Tier 3 registered | features so far: 288


### 1i. Tier 4 -- properties no moment captures, then close out the registry

Hill estimator (sharp tail-index estimate, vs. the crude `tail2`/`tail3` exceedance counts
that already score ~0.54 -- tail-thickness change is genuinely distinct from a variance
change), Vasicek differential-entropy spacings estimator, permutation entropy (ordinal
patterns on the **raw chronological** window -- invisible to the marginal, sensitive to
dependence/dynamics changes only), order-1 ARCH-LM and Ljung-Box (both target volatility
*dynamics*, not just volatility level -- directly aimed at the `e2 > z2` finding: the live axis
may be how the residual's variance evolves, not merely its current value), and a single-scale
R/S Hurst exponent (long-memory shift).

Once these are registered, `FEATURES`/`N_FEATURES` are finally derived from the complete
`REGISTRY` (base + Tier 1-4) -- this is the only place they're computed, never hand-written.
`ACTIVE_FAMILIES` is the speed/selection switch used by `infer()` and the crunch-submission
`train()` (section 4/5): `None` means "every family", a `set` of family `.name`s restricts to
that subset. `resolve_active_families()` validates any such set against `REGISTRY` up front so a
typo'd family name fails loudly instead of silently dropping every feature it would have added.


In [ ]:
class HillEstimatorFamily(TwoSampleFamily):
    """Hill tail-index estimator on |window| vs. |reference|, using k = m//4
    upper order statistics. Standardized via the estimator's known asymptotic
    SE = alpha/sqrt(k)."""
    name = "hill"
    cost = "heavy"

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.ref_alpha = {ch: self._hill(np.abs(self.ref[ch])) for ch in ("z", "e")}

    @staticmethod
    def _hill(absx, k=None):
        n = absx.size
        if n < 20:
            return 4.0
        k = k or max(4, n // 4)
        k = min(k, n - 1)
        s = np.sort(absx)[::-1]
        top = s[:k]
        xk = s[k]
        if xk <= 0:
            return 4.0
        logs = np.log(np.maximum(top / xk, 1e-12))
        m = float(logs.mean())
        return (k / m) if m > 1e-9 else 4.0

    def stat(self, w, r):
        m = w.size
        if m < 20:
            return 0.0
        ch = self._cur_ch
        alpha_ref = self.ref_alpha[ch]
        k = max(4, m // 4)
        alpha_w = self._hill(np.abs(w), k)
        se = alpha_ref / math.sqrt(k)
        return (alpha_w - alpha_ref) / se if se > 0 else 0.0

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            self._cur_ch = ch
            for W in WINDOWS:
                w = self._window(ctx, ch, W)
                out.append(_safe(self.stat(w, self.ref[ch])))
        return out


class DiffEntropyFamily(TwoSampleFamily):
    """Vasicek spacings estimator of differential entropy."""
    name = "entropy"
    cost = "heavy"
    needs_sorted = True

    def init(self, hist: HistoricalRef):
        super().init(hist)
        self.ref_h = {ch: self._vasicek(self.ref[ch]) for ch in ("z", "e")}

    @staticmethod
    def _vasicek(sorted_x):
        m = sorted_x.size
        if m < 8:
            return 0.0
        q = max(1, int(round(math.sqrt(m))))
        idx = np.arange(m)
        lo = np.clip(idx - q, 0, m - 1)
        hi = np.clip(idx + q, 0, m - 1)
        diffs = sorted_x[hi] - sorted_x[lo]
        diffs = np.maximum(diffs, 1e-9)
        return float(np.mean(np.log(m / (2.0 * q) * diffs)))

    def stat(self, w, r):
        m = w.size
        if m < 8:
            return 0.0
        ch = self._cur_ch
        h_w = self._vasicek(w)
        h_r = self.ref_h[ch]
        return (h_w - h_r) * math.sqrt(m)

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            self._cur_ch = ch
            for W in WINDOWS:
                w = ctx.sorted_window(ch, W)
                out.append(_safe(self.stat(w, self.ref[ch])))
        return out


class PermutationEntropyFamily(FeatureFamily):
    """Ordinal-pattern entropy (order 3) of the RAW chronological window --
    invisible to the marginal, sensitive to dynamics/dependence changes. Under
    iid continuous data normalized PE -> 1 exactly, so 1.0 is the reference
    constant (no need to store a separate reference sample)."""
    name = "perm_ent"
    cost = "sorted"
    D = 3

    def names(self):
        return ["perm_ent_%s_w%d" % (ch, W) for ch in self.channels for W in WINDOWS]

    def _pe(self, x: np.ndarray) -> float:
        d = self.D
        m = x.size - d + 1
        if m < 8:
            return 1.0
        win = np.lib.stride_tricks.sliding_window_view(x, d)
        ordi = np.argsort(win, axis=1, kind="stable")
        code = (ordi * (d ** np.arange(d))).sum(axis=1)
        cnt = np.bincount(code)
        cnt = cnt[cnt > 0].astype(np.float64)
        p = cnt / cnt.sum()
        h = -(p * np.log(p)).sum()
        return h / math.log(math.factorial(d))

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            for W in WINDOWS:
                w = ctx.channels[ch].window(W, ctx.t)
                pe = self._pe(w)
                out.append(_safe((pe - 1.0) * math.sqrt(max(w.size, 1))))
        return out


class ArchLMFamily(FeatureFamily):
    """Order-1 ARCH-LM: does squared-e autocorrelate (volatility clustering
    changed), not just its level. Direct target given e2 > z2 in the diagnostic."""
    name = "archlm"
    cost = "sorted"
    channels = ("e",)

    def names(self):
        return ["archlm_%s_w%d" % (ch, W) for ch in self.channels for W in WINDOWS]

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            for W in WINDOWS:
                w = ctx.channels[ch].window(W, ctx.t)
                m = w.size
                if m < 10:
                    out.append(0.0)
                    continue
                sq = w * w
                x, y = sq[:-1], sq[1:]
                if x.std() < 1e-9 or y.std() < 1e-9:
                    out.append(0.0)
                    continue
                r = float(np.corrcoef(x, y)[0, 1])
                lm = (m - 1) * r * r
                out.append(_safe((lm - 1.0) / math.sqrt(2.0)))
        return out


class LjungBoxFamily(FeatureFamily):
    """Reuses the autocorrelation pattern from AutocorrFamily (1d), but tests
    joint significance of lags 1..H rather than one lag at a time."""
    name = "ljungbox"
    cost = "sorted"
    H = 5

    def names(self):
        return ["ljungbox_%s_w%d" % (ch, W) for ch in self.channels for W in WINDOWS]

    def update(self, ctx: Ctx):
        out = []
        for ch in self.channels:
            for W in WINDOWS:
                w = ctx.channels[ch].window(W, ctx.t)
                m = w.size
                h = min(self.H, max(m // 5, 1))
                if m < 12 or h < 1:
                    out.append(0.0)
                    continue
                x = w - w.mean()
                d = float(np.dot(x, x))
                if d <= 0:
                    out.append(0.0)
                    continue
                q = 0.0
                for k in range(1, h + 1):
                    if m - k <= 0:
                        break
                    rho_k = float(np.dot(x[k:], x[:m - k])) / d
                    q += rho_k * rho_k / (m - k)
                q *= m * (m + 2)
                out.append(_safe((q - h) / math.sqrt(2.0 * h)))
        return out


class HurstFamily(TwoSampleFamily):
    """Single-scale R/S estimate of the Hurst exponent (long-memory shift).
    Reference value under iid is exactly 0.5."""
    name = "hurst"
    cost = "heavy"
    needs_sorted = False

    def stat(self, w, r):
        m = w.size
        if m < 16:
            return 0.0
        x = w - w.mean()
        cum = np.cumsum(x)
        rng = float(cum.max() - cum.min())
        s = float(w.std(ddof=1))
        if s <= 0 or rng <= 0:
            return 0.0
        h = math.log(rng / s) / math.log(m)
        return (h - 0.5) * math.sqrt(m)


REGISTRY.extend([
    HillEstimatorFamily(), DiffEntropyFamily(), PermutationEntropyFamily(),
    ArchLMFamily(), LjungBoxFamily(), HurstFamily(),
])

# =====================================================================
#  REGISTRY IS NOW COMPLETE -- derive FEATURES / N_FEATURES from it
# =====================================================================
FEATURES = [nm for f in REGISTRY for nm in f.names()]
N_FEATURES = len(FEATURES)
assert len(set(FEATURES)) == N_FEATURES, "duplicate feature names in REGISTRY"
print("REGISTRY: %d families, %d total features" % (len(REGISTRY), N_FEATURES))

ACTIVE_FAMILIES = {
    # --- legacy: structurally required ---
    "time", "h",                      # t_step/t_log are exactly 0.5 by construction;
                                      # h_* are context, judged by ablation not univariate AUC
    # --- legacy: cleared cutoff ---
    "e2", "z2", "tail",
    # --- new: cleared cutoff, all dispersion tests ---
    "ftest", "levene", "fligner", "lmom", "entropy", "iqr",
    # --- rank-based scale: keep ONE, ablate to pick ---
    "mood",                           # vs "ansari" / "sigtuk"
}
ACTIVE_FAMILIES = resolve_active_families(ACTIVE_FAMILIES)    # None = every family. Set to e.g. {"z2","e2","welch",...} for speed.


def resolve_active_families(active_families):
    """Validate a family-name subset against REGISTRY -- fails loudly on a typo
    instead of silently emitting zero columns for that family."""
    if active_families is None:
        return None
    known = {f.name for f in REGISTRY}
    unknown = set(active_families) - known
    assert not unknown, "unknown family name(s) in ACTIVE_FAMILIES: %s" % sorted(unknown)
    return set(active_families)


ACTIVE_FAMILIES = resolve_active_families(ACTIVE_FAMILIES)


REGISTRY: 42 families, 332 total features


## 2. Building the training matrix

One row per `(series, online step)`. With 10k series x up to 1000 steps that is up to 10M rows,
so we subsample steps on a geometric schedule (dense early, sparse late) and carry an
**inverse-sampling-probability weight** so the fitted probability stays calibrated.

Validation series are kept at **full resolution** — TS-AUC must be evaluated on every step.

`extract_series`/`build_dataset` now take an optional `active_families` override (defaults to
the global `ACTIVE_FAMILIES`). The offline iteration loop (diagnostic, cache, CV/ablate) always
calls with `active_families=None` (full registry) regardless of the global -- see section 2b --
so trying a feature subset is a column-selection problem on an already-cached matrix, never a
re-extraction. The crunch-submission `train()`/`infer()` (sections 4/5) leave it at the global
`ACTIVE_FAMILIES`, so the deployed model only ever pays for the families it actually uses.


In [ ]:
KEEP_EVERY = ((24, 1), (128, 2), (512, 4), (10 ** 9, 8))   # (up_to_t, stride)

_USE_GLOBAL_ACTIVE = object()   # sentinel: "use whatever ACTIVE_FAMILIES is right now"


def _stride_at(t: int) -> int:
    for up_to, stride in KEEP_EVERY:
        if t < up_to:
            return stride
    return KEEP_EVERY[-1][1]


def extract_series(x_hist, x_online, tau=None, subsample=True, active_families=_USE_GLOBAL_ACTIVE):
    """Run the extractor over one series. Returns (X, y, tvec, w).

    active_families: _USE_GLOBAL_ACTIVE (default) resolves to the current
    ACTIVE_FAMILIES global at call time (this is what train()/infer() want --
    fast, submission-path extraction). Pass None explicitly to force the FULL
    registry regardless of the global (this is what the offline cache/diagnostic/
    ablate loop wants -- always extract everything, select columns after)."""
    fams = ACTIVE_FAMILIES if active_families is _USE_GLOBAL_ACTIVE else active_families
    fx = StreamFeatures(x_hist, active_families=fams)
    rows, ys, ts, ws = [], [], [], []
    for t, point in enumerate(x_online):
        f = fx.update(point)
        if subsample:
            stride = _stride_at(t)
            if t % stride != 0:
                continue
            ws.append(float(stride))
        else:
            ws.append(1.0)
        rows.append(list(f))
        ys.append(0 if (tau is None or t < tau) else 1)
        ts.append(t)
    X = np.asarray(rows, dtype=np.float32)
    return X, np.asarray(ys, np.int8), np.asarray(ts, np.int32), np.asarray(ws, np.float32)


def build_dataset(datasets, subsample=True, max_series=None, n_jobs=-1, verbose=1,
                   active_families=_USE_GLOBAL_ACTIVE,backend=None):
    items = list(datasets)[: max_series] if max_series else list(datasets)

    def one(rec):
        did, xh, xo, tau = rec
        X, y, t, w = extract_series(xh, xo, tau, subsample=subsample, active_families=active_families)
        g = np.full(t.shape, did, dtype=np.int64)
        return X, y, t, w, g

    # threading backend: StreamFeatures/FeatureFamily are per-call-safe (each
    # family is .clone()-d off the REGISTRY prototype, see FeatureFamily.clone()),
    # so concurrent series extraction across threads doesn't share mutable state.
    res = joblib.Parallel(n_jobs=n_jobs, verbose=verbose, batch_size=8, backend="loky")(
            joblib.delayed(one)(r) for r in items
    )
    X = np.concatenate([r[0] for r in res])
    y = np.concatenate([r[1] for r in res])
    t = np.concatenate([r[2] for r in res])
    w = np.concatenate([r[3] for r in res])
    g = np.concatenate([r[4] for r in res])
    return X, y, t, w, g


## 2b. Disk cache for the fast iteration loop

The intended workflow is **add a feature -> test it alone -> train on a subset of features ->
repeat**, and extraction (not LightGBM on an existing matrix) is the expensive step. So: always
extract the **full registry** (never a subset -- see `active_families=None` above), cache the
resulting `(X, y, t, w, g, feature_names)` to disk, and have everything downstream -- the
diagnostic, `quick_train`, `ablate` -- work by **selecting columns** out of that cached matrix.
Adding a feature invalidates the cache automatically (the key includes the feature-name list);
changing only `ACTIVE_FAMILIES` does not, since the cache never depends on it.

`select_features(...)` resolves any combination of explicit family names, explicit feature
names, glob patterns, or a `min_ts_auc=` rule against a diagnostic table into column indices --
everything downstream (`quick_train`, `ablate`, `cv_evaluate`) takes its columns from this.


In [78]:
import fnmatch
import hashlib
import json

CACHE_DIR = "cache"

# Exact feature -> family-name map, built directly from REGISTRY (not guessed
# from the string, which breaks for any family whose .name itself contains an
# underscore, e.g. "perm_ent_z_w16" would wrongly split to "perm").
FEATURE_TO_FAMILY = {nm: f.name for f in REGISTRY for nm in f.names()}


def cache_key(feature_names, series_ids, subsample_schedule, seed) -> str:
    payload = json.dumps({
        "features": list(feature_names),
        "series_ids": sorted(str(s) for s in series_ids),
        "schedule": list(subsample_schedule),
        "seed": seed,
    }, sort_keys=True).encode()
    return hashlib.sha256(payload).hexdigest()[:24]


def get_cached_matrix(datasets, subsample=True, max_series=None, seed=0, cache_dir=CACHE_DIR):
    """Always extracts the FULL registry (active_families=None). Cache key covers
    the feature-name list + series-id list + subsample schedule + seed, so adding
    a feature invalidates it automatically; ACTIVE_FAMILIES plays no part."""
    items = list(datasets)[:max_series] if max_series else list(datasets)
    series_ids = [r[0] for r in items]
    key = cache_key(FEATURES, series_ids, KEEP_EVERY if subsample else (), seed)
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, "%s.joblib" % key)
    if os.path.exists(path):
        print("cache hit:", path)
        return joblib.load(path)
    print("cache miss, extracting %d series (full registry) ..." % len(items))
    X, y, t, w, g = build_dataset(items, subsample=subsample, active_families=None)
    payload = (X, y, t, w, g, list(FEATURES))
    joblib.dump(payload, path)
    print("cached ->", path, "shape", X.shape)
    return payload


def clear_cache(cache_dir=CACHE_DIR):
    if os.path.isdir(cache_dir):
        for fn in os.listdir(cache_dir):
            if fn.endswith(".joblib"):
                os.remove(os.path.join(cache_dir, fn))
    print("cache cleared:", cache_dir)


def select_features(feature_names, families=None, features=None, patterns=None,
                     min_ts_auc=None, diag=None):
    """Resolve any combination of family names / explicit feature names / glob
    patterns / a diagnostic-driven min_ts_auc rule into column indices. Returns
    (idx, matching_names). No selector given -> every column."""
    feature_names = list(feature_names)
    chosen = set()
    any_selector = False
    if families:
        any_selector = True
        fams = set(families)
        chosen |= {i for i, nm in enumerate(feature_names) if FEATURE_TO_FAMILY.get(nm) in fams}
    if features:
        any_selector = True
        want = set(features)
        chosen |= {i for i, nm in enumerate(feature_names) if nm in want}
    if patterns:
        any_selector = True
        for pat in patterns:
            chosen |= {i for i, nm in enumerate(feature_names) if fnmatch.fnmatch(nm, pat)}
    if min_ts_auc is not None:
        any_selector = True
        assert diag is not None, "min_ts_auc requires diag= a diagnostic DataFrame (section 3)"
        keep = set(diag.loc[diag["ts_auc_abs"] >= min_ts_auc, "feature"])
        chosen |= {i for i, nm in enumerate(feature_names) if nm in keep}
    if not any_selector:
        chosen = set(range(len(feature_names)))
    idx = np.array(sorted(chosen), dtype=np.int64)
    return idx, [feature_names[i] for i in idx]


## 3. TS-AUC, and a per-feature diagnostic

`ts_auc` is the leaderboard metric, unchanged.

`run_feature_diagnostic` replaces `single_feature_ts_auc`: it samples series **uniformly at
random** (seeded, default 10k) rather than taking the first N, extracts the **full registry**
through the section 2b cache (so re-running after adding one family doesn't re-extract
everything else), and folds each feature's TS-AUC via `max(a, 1-a)` as before -- but now next to
an honest noise floor from `permutation_null_ts_auc`, because the raw folded number cannot be
thresholded on its own. Two reasons:

1. **Folding bias.** `max(a, 1-a)` does not center a genuinely null feature on 0.50; sampling
   noise pushes it up in both tails and only the upper one survives the fold, so null features
   look positive on average.
2. **Effective n is series count, not row count.** The same series reappears in every `t`-slice,
   so per-slice AUCs for one series are strongly correlated -- hundreds of thousands of rows
   buy far fewer independent observations than that number suggests.

`permutation_null_ts_auc` shuffles **series-level** break labels (which series get treated as
breaking), preserving each series' own row/`t` structure -- never shuffling rows independently,
which would destroy exactly the within-series correlation that inflates the naive null. It
reports the mean and the 95th percentile of the folded AUC under that null; the P95 is printed
as `CUTOFF` and used as the `above_null` column in the results table.


In [79]:
def ts_auc(y, t, score):
    y = np.asarray(y); t = np.asarray(t); score = np.asarray(score, dtype=np.float64)
    order = np.argsort(t, kind="mergesort")
    y, t, score = y[order], t[order], score[order]
    bounds = np.flatnonzero(np.diff(t)) + 1
    num = den = 0.0
    for lo, hi in zip(np.r_[0, bounds], np.r_[bounds, len(t)]):
        yy = y[lo:hi]
        npos = int(yy.sum()); nneg = len(yy) - npos
        if npos == 0 or nneg == 0:
            continue
        w = float(npos * nneg)
        num += w * roc_auc_score(yy, score[lo:hi])
        den += w
    return num / den if den > 0 else 0.5


def _folded_ts_auc_table(X, y, t, feature_names):
    rows = []
    for k, nm in enumerate(feature_names):
        s = np.nan_to_num(X[:, k].astype(np.float64), nan=0.0, posinf=0.0, neginf=0.0)
        a = ts_auc(y, t, s)
        rows.append((nm, max(a, 1 - a), a))
    return pd.DataFrame(rows, columns=["feature", "ts_auc_abs", "ts_auc_raw"])


def permutation_null_ts_auc(X, y, t, g, feature_idx, feature_names, n_perm=100, seed=0):
    """Series-level label permutation: shuffle which SERIES are treated as
    breaking, keep each series' own row/t structure intact. A relabeled
    "breaker" reuses its own original first-break t as a synthetic tau (so the
    permuted series still has a well-defined post-break region); a real
    breaker relabeled non-breaking has all its rows set to y=0. Never shuffle
    rows independently -- that destroys the within-series correlation that is
    exactly why the naive (row-count) null is too tight."""
    rng = np.random.default_rng(seed)
    uniq_series = np.unique(g)
    series_breaks = {}
    for sid in uniq_series:
        series_breaks[sid] = int(y[g == sid].max())
    sids = list(series_breaks.keys())
    breaks_arr = np.array([series_breaks[s] for s in sids])

    null_aucs = {nm: [] for nm in feature_names}
    for _ in range(n_perm):
        perm_breaks = rng.permutation(breaks_arr)
        perm_map = dict(zip(sids, perm_breaks))
        y_perm = np.empty_like(y)
        for sid in uniq_series:
            mask = g == sid
            if perm_map[sid] == 1 and series_breaks[sid] == 1:
                y_perm[mask] = y[mask]
            elif perm_map[sid] == 1 and series_breaks[sid] == 0:
                tt = t[mask]
                cut = tt[len(tt) // 2] if len(tt) else 0
                y_perm[mask] = (tt >= cut).astype(y.dtype)
            else:
                y_perm[mask] = 0
        for j, nm in zip(feature_idx, feature_names):
            s = np.nan_to_num(X[:, j].astype(np.float64), nan=0.0, posinf=0.0, neginf=0.0)
            a = ts_auc(y_perm, t, s)
            null_aucs[nm].append(max(a, 1 - a))

    rows = [(nm, float(np.mean(v)), float(np.percentile(v, 95))) for nm, v in null_aucs.items()]
    return pd.DataFrame(rows, columns=["feature", "null_mean", "null_p95"])


def run_feature_diagnostic(train_data, seed=0, n_series=10_000, n_perm=100,
                            perm_features=None, cache_dir=CACHE_DIR):
    """Sample n_series uniformly at random, extract the full registry (through
    the section 2b cache), fold TS-AUC per feature, and attach an honest
    permutation-null cutoff. Returns (results_df, family_rollup_df, cutoff,
    (X, y, t, w, g, feature_names)) -- the last element is the raw cached matrix
    for the sampled series, handed back so downstream cells (e.g. the per-feature
    AUC-by-age plot at the bottom of the notebook) don't need a second extraction."""
    items = list(train_data)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(items), size=min(n_series, len(items)), replace=False)
    sample = [items[i] for i in idx]

    X, y, t, w, g, feature_names = get_cached_matrix(sample, subsample=True, seed=seed,
                                                       cache_dir=cache_dir)
    diag = _folded_ts_auc_table(X, y, t, feature_names)
    diag["n_series"] = len(sample)
    diag["n_rows"] = X.shape[0]
    diag["family"] = diag["feature"].map(FEATURE_TO_FAMILY)

    # permutation null on a representative subset (best feature per family)
    # unless the caller asks for a specific list -- running it on all ~1500
    # features is possible but slow; the family-representative set is enough
    # to set CUTOFF.
    if perm_features is None:
        rep = diag.sort_values("ts_auc_abs", ascending=False).drop_duplicates(subset="family")
        perm_features = rep["feature"].tolist()
    perm_idx = [feature_names.index(nm) for nm in perm_features]
    null_df = permutation_null_ts_auc(X, y, t, g, perm_idx, perm_features, n_perm=n_perm, seed=seed)

    cutoff = float(null_df["null_p95"].max())
    diag = diag.merge(null_df[["feature", "null_mean", "null_p95"]], on="feature", how="left")
    diag["above_null"] = diag["ts_auc_abs"] >= cutoff
    diag = diag.sort_values("ts_auc_abs", ascending=False).reset_index(drop=True)

    rollup = (diag.groupby("family")["ts_auc_abs"]
              .agg(["max", "mean"]).sort_values("max", ascending=False).reset_index())

    print("permutation-null CUTOFF (P95 over %d representative features, %d perms): %.5f"
          % (len(perm_features), n_perm, cutoff))
    return diag, rollup, cutoff, (X, y, t, w, g, feature_names)


### Diagnostic run: 10k series, seeded, with a permutation-null cutoff

Interpretation rules (read before touching the results table):

- **`t_step`/`t_log` are structurally 0.5 and are always kept.** A pure function of `t` is
  constant within a `t`-slice, so TS-AUC cannot see it at all -- this is expected, not a bug,
  and it says nothing about whether `t` helps through interactions (it does, via the tree
  splits LightGBM builds on it jointly with other features).
- **`h_*` context features carry no break evidence by design** (design principle #5, section 1)
  -- they let the model learn how much to trust each evidence feature for this kind of process.
  They pay off only through interactions, so they are judged by ablation (section 3c), never by
  this table.
- **Univariate AUC cannot see complementarity.** A feature at 0.51 that is uncorrelated with the
  `e2_*` block may add real value to the model; a feature at 0.55 that is 0.95-correlated with
  `e2_ew0.08` adds nothing once `e2_ew0.08` is already in. **Final keep/drop decisions are made
  by CV ablation (section 3c), not by this table** -- this table is for triage (what's worth
  ablating) and for measuring the honest noise floor via the permutation null, not a keep/drop
  rule on its own.


In [80]:
import time

def bench_extractor(datasets, n_series=3):
    items = list(datasets)[:n_series]
    base = {f.name for f in REGISTRY if f.cost == "O1"}
    def t_of(rec, active):
        xh, xo = rec[1], rec[2]
        fx = StreamFeatures(xh, active_families=active)
        t0 = time.perf_counter()
        for v in xo: fx.update(v)
        return time.perf_counter() - t0
    b = np.mean([t_of(r, base) for r in items])
    full = np.mean([t_of(r, None) for r in items])
    print("O1 baseline : %6.3f s/series" % b)
    print("FULL registry: %6.3f s/series" % full)
    print("-> 10k series, 4 procs: ~%.1f h\n" % (full * 10000 / 4 / 3600))
    rows = sorted(((np.mean([t_of(r, base | {f.name}) for r in items]) - b, f.name, f.cost)
                   for f in REGISTRY if f.cost != "O1"), reverse=True)
    for d, n, c in rows[:12]:
        print("  %-12s %-7s %7.3f s" % (n, c, d))

bench_extractor(train_data)

O1 baseline :  0.035 s/series
FULL registry:  3.017 s/series
-> 10k series, 4 procs: ~2.1 h

  mmd          heavy     0.459 s
  fligner      sorted    0.280 s
  bowley       sorted    0.149 s
  iqr          sorted    0.146 s
  levene       sorted    0.145 s
  moors        sorted    0.144 s
  energy       heavy     0.130 s
  wass         heavy     0.122 s
  ansari       sorted    0.106 s
  mood         sorted    0.105 s
  perm_ent     sorted    0.102 s
  mwu          sorted    0.100 s


In [81]:
DIAG_SEED = 0
DIAG_N_SERIES = 2500

diag, family_rollup, CUTOFF, diag_cache = run_feature_diagnostic(
    train_data, seed=DIAG_SEED, n_series=DIAG_N_SERIES, n_perm=100,
)
Xd, yd, td, wd, gd, FEATURES_D = diag_cache   # full-registry matrix for the sampled series --
                                               # reusable by quick_train/ablate (section 3c) and
                                               # by plot_feature_auc_by_d (bottom of notebook)
print("CUTOFF (permutation-null P95):", CUTOFF)
diag.head(30)


cache miss, extracting 2500 series (full registry) ...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done 344 tasks      | elapsed: 10.5min
[Parallel(n_jobs=-1)]: Done 1544 tasks      | elapsed: 45.6min
[Parallel(n_jobs=-1)]: Done 2500 out of 2500 | elapsed: 73.1min finished


cached -> cache/92275c3676027abbf0228b12.joblib shape (383474, 332)
permutation-null CUTOFF (P95 over 42 representative features, 100 perms): 0.53036
CUTOFF (permutation-null P95): 0.5303635792382227


,feature,ts_auc_abs,ts_auc_raw,n_series,n_rows,family,null_mean,null_p95,above_null
0,e2_ew0.02,0.559797,0.559797,2500,383474,e2,0.514398,0.527436,True
1,e2_w256,0.559673,0.559673,2500,383474,e2,NaN,NaN,True
2,ftest_e_w256,0.559580,0.559580,2500,383474,ftest,0.514938,0.530364,True
3,lmom_lkurt_e_w32,0.559081,0.559081,2500,383474,lmom,0.514770,0.528617,True
4,levene_e_w256,0.558717,0.558717,2500,383474,levene,0.514675,0.529543,True
5,e2_exp,0.558482,0.558482,2500,383474,e2,NaN,NaN,True
6,fligner_e_w256,0.558101,0.558101,2500,383474,fligner,0.514620,0.529175,True
7,mood_e_w256,0.557684,0.557684,2500,383474,mood,0.513970,0.526859,True
8,e2_w64,0.556217,0.556217,2500,383474,e2,NaN,NaN,True
9,ftest_e_w64,0.555930,0.555930,2500,383474,ftest,NaN,NaN,True


In [82]:
# Where does the signal live, by feature family? (returned by run_feature_diagnostic,
# shown here for convenience) -- families whose max ts_auc_abs clears CUTOFF are the
# ones worth ablating in section 3c; below it, they're indistinguishable from noise
# at this sample size.
family_rollup["clears_cutoff"] = family_rollup["max"] >= CUTOFF
family_rollup.head(80)


,family,max,mean,clears_cutoff
0,e2,0.559797,0.546547,True
1,ftest,0.559580,0.542898,True
2,lmom,0.559081,0.516125,True
3,levene,0.558717,0.541025,True
4,fligner,0.558101,0.540245,True
5,mood,0.557684,0.537955,True
6,ansari,0.554890,0.536236,True
7,sigtuk,0.554858,0.536222,True
8,entropy,0.554736,0.536893,True
9,tail,0.544135,0.533122,True


## 3b. K-fold cross-validation over series

The single `train()` call above reports TS-AUC on **one** random 15% hold-out of series.
That number has real sampling noise — with a few thousand series and a handful of break
types, a single split can favor or disfavor a given feature/param change purely by luck of
which series ended up in validation.

`cv_evaluate()` below repeats the same train/validate procedure `n_folds` times, each time
holding out a *different* group of series, and reports the **mean and spread** of TS-AUC
across folds. Only trust a change (new feature, pruned family, different LGB params) if it
moves the CV mean by more than roughly the fold-to-fold std — otherwise you're chasing noise.

Cost: this is `n_folds` full training runs, so ~5x the time of a single `train()` call.
Use `max_series` to iterate fast on a subset first, then run the full thing to confirm
before you commit to a change.

**Extended for the fast iteration loop (section 3c):** `cv_evaluate` now optionally takes a
pre-extracted `X, y, t, g` (e.g. from `get_cached_matrix`) plus a `feature_idx` column
selection, in which case it skips `build_dataset` entirely and folds directly by the unique
ids already present in `g`. This is what lets `ablate()` re-run CV under a different feature
selection in seconds -- the original `datasets=`-based call (re-extracts from raw series) still
works unchanged for anything not going through the cache.


In [83]:
def _fit_and_score(Xtr, ytr, wtr, feature_names, Xva, yva, tva, num_rounds=None):
    dtr = lgb.Dataset(Xtr, label=ytr, weight=wtr, feature_name=feature_names, free_raw_data=False)
    dva = lgb.Dataset(Xva, label=yva, feature_name=feature_names, reference=dtr, free_raw_data=False)
    booster = lgb.train(
        LGB_PARAMS, dtr, num_boost_round=num_rounds or NUM_ROUNDS,
        valid_sets=[dva], valid_names=["valid"],
        callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(0)],
    )
    pva = booster.predict(Xva, num_iteration=booster.best_iteration)
    return ts_auc(yva, tva, pva)


def cv_evaluate(datasets=None, n_folds: int = 5, seed: int = 0,
                 max_series=None, num_rounds=None, verbose: bool = True,
                 X=None, y=None, t=None, g=None, feature_idx=None, feature_names=None):
    """K-fold cross-validation over series.

    Two modes:
    - `datasets=` (original behavior): re-extracts each fold from raw series via
      build_dataset (train subsampled, valid at full resolution -- the fidelity-
      first path, what train() itself uses).
    - `X=, y=, t=, g=` (fast-iteration mode, section 3c): works directly off an
      already-extracted matrix (e.g. from get_cached_matrix), optionally
      restricted to `feature_idx` columns. NOTE: both "train" and "valid" rows
      here share whatever subsample schedule the cache was built with (unlike
      the datasets= path, which validates at full resolution) -- this trades a
      little fidelity for speed, which is the point of this path. Confirm a
      real decision with the datasets= path or crunch_tools.test() before
      committing to it.
    """
    if X is not None:
        assert y is not None and t is not None and g is not None, \
            "matrix mode requires X, y, t, g"
        names = list(feature_names) if feature_names is not None else list(FEATURES)
        if feature_idx is not None:
            Xsel = X[:, feature_idx]
            names = [names[i] for i in feature_idx]
        else:
            Xsel = X

        uniq = np.unique(g)
        if max_series:
            uniq = uniq[:max_series]
            mask = np.isin(g, uniq)
            Xsel, y, t, g = Xsel[mask], y[mask], t[mask], g[mask]
            uniq = np.unique(g)

        rng = np.random.default_rng(seed)
        perm = rng.permutation(len(uniq))
        folds = np.array_split(perm, n_folds)

        fold_aucs = []
        for k in range(n_folds):
            va_ids = set(uniq[folds[k]].tolist())
            va_mask = np.isin(g, list(va_ids))
            tr_mask = ~va_mask
            auc = _fit_and_score(Xsel[tr_mask], y[tr_mask], np.ones(tr_mask.sum(), dtype=np.float32),
                                  names, Xsel[va_mask], y[va_mask], t[va_mask], num_rounds)
            fold_aucs.append(auc)
            if verbose:
                print("fold %d/%d | train %5d rows | valid %5d rows | TS-AUC %.5f"
                      % (k + 1, n_folds, int(tr_mask.sum()), int(va_mask.sum()), auc))
    else:
        items = list(datasets)
        if max_series:
            items = items[:max_series]

        n = len(items)
        rng = np.random.default_rng(seed)
        perm = rng.permutation(n)
        folds = np.array_split(perm, n_folds)

        fold_aucs = []
        for k in range(n_folds):
            va_idx = set(folds[k].tolist())
            tr_items = [items[i] for i in range(n) if i not in va_idx]
            va_items = [items[i] for i in folds[k]]

            Xtr, ytr, ttr, wtr, _ = build_dataset(tr_items, subsample=True, verbose=0)
            Xva, yva, tva, _, _ = build_dataset(va_items, subsample=False, verbose=0)

            auc = _fit_and_score(Xtr, ytr, wtr, FEATURES, Xva, yva, tva, num_rounds)
            fold_aucs.append(auc)
            if verbose:
                print("fold %d/%d | train %4d series | valid %4d series | TS-AUC %.5f"
                      % (k + 1, n_folds, len(tr_items), len(va_items), auc))

    fold_aucs = np.array(fold_aucs)
    if verbose:
        print("\nCV TS-AUC: %.5f +/- %.5f  (min %.5f, max %.5f)"
              % (fold_aucs.mean(), fold_aucs.std(), fold_aucs.min(), fold_aucs.max()))
    return fold_aucs


## 3c. The fast iteration loop: `quick_train` + `ablate`

This is the actual point of the refactor. The intended workflow is **add a feature -> test it
alone (section 3, diagnostic) -> train on a subset of features -> repeat**, and it should cost
seconds per iteration, not a re-extraction.

- `quick_train(cache, selection, ...)`: a thin wrapper over the same fit logic as `train()`
  (same `LGB_PARAMS`, same series-level split), but takes a cached matrix + column selection
  directly. No re-extraction, no disk writes, no model artifact -- just a hold-out TS-AUC number.
- `ablate(cache, baseline_selection, ...)`: given a baseline feature selection, drops (or adds)
  each candidate family in turn and re-runs `cv_evaluate` (matrix mode) on the cached matrix,
  returning `delta_mean`/`delta_std` per family vs. the baseline CV. **This is the tool that
  makes keep/drop decisions** -- it's the only one of these that can see complementarity
  (section 3's diagnostic can't: a 0.51 feature uncorrelated with the `e2_*` block may help even
  though a 0.55 feature 0.95-correlated with it won't).

**Accept-rule:** only accept a change (new feature, dropped family, different LGB params) if it
moves the **CV mean** by more than the **fold-to-fold std**. A single train()/quick_train split
difference of 0.002-0.005 is noise, not signal -- and `y_test.reduced.parquet` (section 6) is
about 1% the size of train, so it's a smoke test for the submission plumbing, never the metric
you make a keep/drop call on.


In [84]:
def quick_train(cache, selection, max_series=None, num_rounds=None, seed=0, valid_fraction=None):
    """Thin wrapper over the same fit logic as train(), on an already-cached
    matrix + column selection. No re-extraction, no disk writes, no artifact --
    just a hold-out TS-AUC. `selection` is either an index array or a
    (idx, names) tuple, e.g. straight from select_features(...)."""
    X, y, t, w, g, feature_names = cache
    idx = selection[0] if isinstance(selection, tuple) else np.asarray(selection)
    names = [feature_names[i] for i in idx]
    Xsel = X[:, idx]

    uniq = np.unique(g)
    if max_series:
        uniq = uniq[:max_series]
        mask = np.isin(g, uniq)
        Xsel, y, t, w, g = Xsel[mask], y[mask], t[mask], w[mask], g[mask]
        uniq = np.unique(g)

    vf = valid_fraction if valid_fraction is not None else VALID_FRACTION
    rng = np.random.default_rng(seed)
    n_val = max(1, int(round(len(uniq) * vf)))
    val_ids = set(rng.choice(uniq, size=n_val, replace=False).tolist())
    va_mask = np.isin(g, list(val_ids))
    tr_mask = ~va_mask

    return _fit_and_score(Xsel[tr_mask], y[tr_mask], w[tr_mask], names,
                           Xsel[va_mask], y[va_mask], t[va_mask], num_rounds)


def ablate(cache, baseline_idx, candidates="families", n_folds=5, seed=0,
           num_rounds=None, max_series=None, verbose=True):
    """For each candidate family, try dropping it (if in the baseline selection)
    or adding it (if not), re-run cv_evaluate on the cached matrix, and report
    the delta vs. the baseline CV mean. This is the tool that makes keep/drop
    decisions -- it sees complementarity that the univariate diagnostic (section
    3) cannot. `baseline_idx` is a column-index array, e.g. from select_features."""
    X, y, t, w, g, feature_names = cache
    name_to_idx = {nm: i for i, nm in enumerate(feature_names)}
    baseline_idx = np.asarray(baseline_idx)
    baseline_names = {feature_names[i] for i in baseline_idx}
    baseline_families = {FEATURE_TO_FAMILY[nm] for nm in baseline_names}

    if candidates == "families":
        candidate_families = sorted({f.name for f in REGISTRY} - {"time"})   # t_step/t_log always kept
    else:
        candidate_families = list(candidates)

    base_fold = cv_evaluate(X=X, y=y, t=t, g=g, feature_idx=baseline_idx, feature_names=feature_names,
                             n_folds=n_folds, seed=seed, num_rounds=num_rounds, max_series=max_series,
                             verbose=False)
    base_mean, base_std = float(base_fold.mean()), float(base_fold.std())
    if verbose:
        print("baseline CV TS-AUC: %.5f +/- %.5f (%d features)\n"
              % (base_mean, base_std, len(baseline_idx)))

    rows = [("__baseline__", base_mean, base_std, 0.0, len(baseline_idx))]
    for fam in candidate_families:
        in_baseline = fam in baseline_families
        if in_baseline:
            mod_names = {nm for nm in baseline_names if FEATURE_TO_FAMILY[nm] != fam}
            action = "drop"
        else:
            fam_names = {nm for nm in feature_names if FEATURE_TO_FAMILY[nm] == fam}
            mod_names = baseline_names | fam_names
            action = "add"
        if not mod_names:
            continue
        mod_idx = np.array(sorted(name_to_idx[nm] for nm in mod_names), dtype=np.int64)
        fold_aucs = cv_evaluate(X=X, y=y, t=t, g=g, feature_idx=mod_idx, feature_names=feature_names,
                                 n_folds=n_folds, seed=seed, num_rounds=num_rounds, max_series=max_series,
                                 verbose=False)
        mean_, std_ = float(fold_aucs.mean()), float(fold_aucs.std())
        delta = mean_ - base_mean
        label = "%s %s" % (action, fam)
        rows.append((label, mean_, std_, delta, len(mod_idx)))
        if verbose:
            print("%-16s | CV %.5f +/- %.5f | delta %+.5f | n_features %d"
                  % (label, mean_, std_, delta, len(mod_idx)))

    return pd.DataFrame(rows, columns=["change", "cv_mean", "cv_std", "delta_mean", "n_features"])


In [85]:
# Example usage of the full loop, using the diagnostic's cached matrix (Xd/yd/td/wd/gd,
# section 3) so nothing here re-extracts:
#
#   baseline_idx, baseline_names = select_features(
#       FEATURES_D, min_ts_auc=CUTOFF, diag=diag)   # everything that cleared the null,
#                                                     # plus t_step/t_log/h_* need adding back
#                                                     # explicitly since they're exempt from
#                                                     # the cutoff rule (see section 3 markdown)
#   baseline_idx, baseline_names = select_features(
#       FEATURES_D, families=set(baseline_names_families) | {"time", "h"},
#       patterns=None)   # (illustrative -- build the real baseline however fits your workflow)
#
#   quick_train((Xd, yd, td, wd, gd, FEATURES_D), (baseline_idx, baseline_names), max_series=2000)
#
#   ablate((Xd, yd, td, wd, gd, FEATURES_D), baseline_idx, candidates="families", n_folds=5)
#
# ablate()'s output table (change, cv_mean, cv_std, delta_mean, n_features) is what decides
# keep/drop -- per the accept-rule above, only act on a delta_mean that exceeds cv_std.
print("quick_train / ablate defined -- see the commented example above for usage.")


quick_train / ablate defined -- see the commented example above for usage.


## 4. `train()`

Binary logloss, group-split by series id. Early stopping on validation logloss, then the real
TS-AUC is reported on the held-out series at full step resolution.

Knobs worth sweeping first: `num_leaves`, `min_data_in_leaf`, and the training-row weight
scheme (`USE_METRIC_WEIGHTS`).

**Train/serve parity, section 5's real risk.** `train()` here is the crunch-submission path, so
it extracts through `build_dataset`'s default (`active_families=_USE_GLOBAL_ACTIVE`, i.e.
whatever `ACTIVE_FAMILIES` currently is) -- the same subset `infer()` will compute, not the full
registry. This is deliberate: `ACTIVE_FAMILIES` is decided offline via the section 3c iteration
loop (which always works off the full-registry cache), and once decided, `train()`/`infer()`
both only ever pay for that subset -- keeping them fast and, more importantly, keeping them
*the same feature set*, which is the actual train/serve-skew risk this refactor introduces (a
model trained on one family list must never be scored with a `StreamFeatures` built for a
different one). The artifact now stores the resolved active-family list and the exact feature
list, and `infer()` reconstructs its extractor from *that*, not from whatever the notebook's
global happens to be at inference time -- see the parity-check cell at the end of section 5.


In [86]:
LGB_PARAMS = dict(
    objective="binary",
    learning_rate=0.04,
    num_leaves=63,
    min_data_in_leaf=300,
    feature_fraction=0.70,
    bagging_fraction=0.80,
    bagging_freq=1,
    lambda_l2=5.0,
    max_bin=127,
    num_threads=max(1, (os.cpu_count() or 4) - 1),
    verbosity=-1,
)
NUM_ROUNDS = 1200
VALID_FRACTION = 0.15
TRAIN_MAX_SERIES = None      # set an int for fast iteration


def active_feature_names(active_families=None) -> List[str]:
    """The feature name list StreamFeatures(..., active_families=active_families)
    actually emits, in order -- FEATURES itself only covers active_families=None
    (the full registry). train()/infer() must always pass their booster's
    feature_name= this, never the bare FEATURES global, whenever ACTIVE_FAMILIES
    is a strict subset."""
    return [nm for f in REGISTRY if active_families is None or f.name in active_families
            for nm in f.names()]


def train(datasets, model_directory_path: str):
    items = list(datasets)
    if TRAIN_MAX_SERIES:
        items = items[:TRAIN_MAX_SERIES]

    ids = np.array([r[0] for r in items])
    rng = np.random.default_rng(0)
    is_val = rng.random(len(items)) < VALID_FRACTION

    tr_items = [r for r, v in zip(items, is_val) if not v]
    va_items = [r for r, v in zip(items, is_val) if v]
    print("train series %d | valid series %d" % (len(tr_items), len(va_items)))

    # active_families left at the sentinel default -> resolves to the current
    # ACTIVE_FAMILIES global, so this is exactly what infer() will compute too.
    Xtr, ytr, ttr, wtr, _ = build_dataset(tr_items, subsample=True)
    Xva, yva, tva, _, _ = build_dataset(va_items, subsample=False)
    print("rows: train %s  valid %s" % (Xtr.shape, Xva.shape))

    feat_names = active_feature_names(ACTIVE_FAMILIES)
    assert Xtr.shape[1] == len(feat_names), (
        "extracted %d columns but active_feature_names() expects %d -- "
        "ACTIVE_FAMILIES changed between extraction and here?" % (Xtr.shape[1], len(feat_names)))

    dtr = lgb.Dataset(Xtr, label=ytr, weight=wtr, feature_name=feat_names, free_raw_data=False)
    dva = lgb.Dataset(Xva, label=yva, feature_name=feat_names, reference=dtr, free_raw_data=False)

    booster = lgb.train(
        LGB_PARAMS, dtr, num_boost_round=NUM_ROUNDS,
        valid_sets=[dva], valid_names=["valid"],
        callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(100)],
    )

    pva = booster.predict(Xva, num_iteration=booster.best_iteration)
    print("HOLD-OUT TS-AUC: %.5f" % ts_auc(yva, tva, pva))

    imp = pd.Series(booster.feature_importance("gain"), index=feat_names).sort_values(ascending=False)
    print(imp.head(25))

    # Model artifact is self-describing: it carries the exact active-family set
    # and feature list it was trained with, so infer() can reconstruct the same
    # extractor from the artifact itself rather than trusting the notebook's
    # ACTIVE_FAMILIES global at inference time (which could have drifted).
    joblib.dump(
        {
            "booster": booster,
            "features": feat_names,
            "active_families": sorted(ACTIVE_FAMILIES) if ACTIVE_FAMILIES else None,
            "best_iteration": booster.best_iteration,
        },
        os.path.join(model_directory_path, "model.joblib"),
    )


## 5. `infer()`

**Watch the time budget.** Up to 10M scores means the per-step cost of `booster.predict` on a
single row (~50-100 us) dominates everything. Two safe levers:

- `PREDICT_STRIDE`: recompute the LightGBM prediction only every `K` steps, where `K` depends
  **only on `t`**. Because every series shares the schedule, all series at step `t` are equally
  stale, so within-`t` ranking is unaffected — this is free speed at large `t`.
- Keep `best_iteration` modest (300-600 trees) and `num_threads=1` inside `infer` when running
  with `INFER_PARALLELISM > 1`.

`SMOOTH_ALPHA` / `RUNNING_MAX` are post-processing switches worth ablating: the label is
monotone non-decreasing in `t` for a given series, so removing downward noise sometimes helps.

**`infer()` reconstructs its extractor from the artifact, not from the notebook's live
`ACTIVE_FAMILIES`.** It loads `art["active_families"]` and builds `StreamFeatures(x_historical,
active_families=art["active_families"])` -- so a stale/edited global in this notebook can never
silently desync what a *loaded* model actually predicts on. This also means `infer()` only ever
computes the family subset the model was trained with (never the full registry), which is what
keeps it inside the prediction time budget regardless of how large the registry grows.


In [87]:
SMOOTH_ALPHA = 0.0     # 0 = off; e.g. 0.3 = EWMA-smooth the score across t
RUNNING_MAX = False    # enforce monotone non-decreasing score


def _stride_for(t: int) -> int:
    if t < 64:
        return 1
    if t < 256:
        return 2
    return 4


def infer(datasets, model_directory_path: str):
    art = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    booster = art["booster"]
    best_it = art["best_iteration"]
    active_families = set(art["active_families"]) if art.get("active_families") else None
    n_feat = len(art["features"])
    booster.params["num_threads"] = 1

    buf = np.empty((1, n_feat), dtype=np.float64)

    yield  # signal readiness

    for x_historical, x_online in datasets:
        fx = StreamFeatures(x_historical, active_families=active_families)
        prev = 0.0
        smoothed = 0.0
        running = 0.0

        for t, point in enumerate(x_online):
            f = fx.update(point)

            if t % _stride_for(t) == 0:
                buf[0, :] = f
                prev = float(booster.predict(buf, num_iteration=best_it)[0])
            p = prev

            if SMOOTH_ALPHA > 0.0:
                smoothed = (1.0 - SMOOTH_ALPHA) * smoothed + SMOOTH_ALPHA * p if t else p
                p = smoothed
            if RUNNING_MAX:
                running = p if p > running else running
                p = running

            if p < 0.0:
                p = 0.0
            elif p > 1.0:
                p = 1.0
            yield float(p)


### Train/infer parity check -- run this after training a model, before trusting `infer()`

This is the actual train/serve-skew risk the registry refactor introduces (the old code used
one hardcoded `StreamFeatures`, so this couldn't drift; now the extractor is parameterized by
`active_families`, so it's an explicit thing to verify rather than assume). Loads the just-
trained artifact, builds a `StreamFeatures` the way `infer()` does (from `art["active_families"]`)
and the way `train()`'s extraction path does (via `active_feature_names(ACTIVE_FAMILIES)` +
`build_dataset`'s default), and asserts they produce identical feature vectors on the same
handful of series.


In [88]:
def check_train_infer_parity(model_directory_path, datasets, n_series=20):
    art = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    infer_active = set(art["active_families"]) if art.get("active_families") else None
    train_active = ACTIVE_FAMILIES
    assert infer_active == train_active, (
        "artifact's active_families (%s) != current ACTIVE_FAMILIES (%s) -- "
        "you're checking parity against the wrong global; re-set ACTIVE_FAMILIES "
        "to what this artifact was trained with." % (infer_active, train_active))
    assert art["features"] == active_feature_names(infer_active), (
        "artifact's stored feature list doesn't match active_feature_names(infer_active) -- "
        "REGISTRY must have changed since this model was trained.")

    items = list(datasets)[:n_series]
    max_abs_diff = 0.0
    for did, xh, xo, tau in items:
        fx_infer = StreamFeatures(xh, active_families=infer_active)     # infer()'s construction
        fx_train = StreamFeatures(xh, active_families=train_active)     # train()'s construction
        for point in xo:
            r_i = np.asarray(fx_infer.update(point), dtype=np.float64)
            r_t = np.asarray(fx_train.update(point), dtype=np.float64)
            max_abs_diff = max(max_abs_diff, float(np.max(np.abs(r_i - r_t))))
            assert np.allclose(r_i, r_t, atol=1e-9, rtol=1e-9), (
                "train/infer feature mismatch for series %s" % did)
    print("PARITY OK over %d series, max abs diff = %.3e" % (len(items), max_abs_diff))


# Run after train() (section 4) / crunch_tools.test() (section 6) has produced model.joblib.
# Fill in the actual model_directory_path crunch used (crunch_tools.test() prints/creates it,
# commonly "resources" or wherever `train()`'s model_directory_path argument pointed):
# check_train_infer_parity("resources", train_data, n_series=20)


In [89]:
# @crunch/keep:on
INFER_PARALLELISM = 1


## 6. Local test + leaderboard metric


In [90]:
crunch_tools.test(
    # force_first_train=False,
    # no_determinism_check=True,
)


17:21:31 
17:21:31 started
17:21:31 running local test
17:21:31 internet access isn't restricted, no check will be done
17:21:31 
17:21:32 executing - command=train


data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test.reduced.parquet (106299 bytes)
data/y_test.reduced.parquet: already exists, file length match
data/y_test_index.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
Exception in initializer:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py", line 434, in _process_worker
    initializer(*initargs)
  File "/usr/local/lib/python3.12/dist-packages/crunch/monkey_patches.py", line 298, in process_initializer
    module = load_user_code(main_file_path, user_code_module_name)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/crunch/command/test.py", line 24, in load_user_code
    spec.loader.exec_module(module)
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1132, in get_code
  File "<frozen importlib._bootstrap_external>", line 1190, in get_data
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/main.py'
Exception in initializer:
Traceback (

TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {EXIT(0), EXIT(0), EXIT(0), EXIT(0)}
Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

In [ ]:
prediction = pd.read_parquet("prediction/prediction.parquet")
y_test = pd.read_parquet("data/y_test.reduced.parquet")

merged = prediction.merge(y_test, how="left", left_index=True, right_index=True)
merged["time_online"] = merged.groupby("id").cumcount()

print("Local TS-AUC: %.5f" % ts_auc(
    merged["target"].values, merged["time_online"].values, merged["prediction"].values
))


## 7. Where to go next

Ordered by expected payoff per unit of effort.

1. **Prune with the diagnostic, then re-densify.** Drop families whose best single-feature
   TS-AUC is ~0.50 and spend the budget on more scales inside the families that survived.

2. **Replace the analytic null with a bootstrap null.** Right now features are standardized
   assuming iid PIT-normal under H0, which is wrong whenever the series is serially dependent.
   Circular-block-bootstrap the historical segment ~100 times, run the extractor on each,
   and store the per-`t` mean/sd of every statistic. Then emit `(stat - mu_null(t)) / sd_null(t)`.
   This makes the features pivotal *for real* and is, in my experience, the single biggest
   available gain in this setup. Cost is O(n_H x B) once per series — vectorize it.

3. **Exact GLR instead of CUSUM.** `max_k sqrt(t-k) * |mean over (k,t]|` is the likelihood-ratio
   statistic for an unknown-tau mean shift. The Page-CUSUM family approximates it; a pruned
   candidate-set implementation gets you the real thing in amortized O(1).

4. **Speed.** The extractor is plain-Python scalars precisely so it can be `@njit`-ed with almost
   no rewrite — move the state into flat float64 arrays and jit `update`. Expect 30-80x.
   If prediction is the bottleneck, distill the booster into a shallow one or export to
   a numpy tree-traversal.

5. **Two-model split.** Break type is heterogeneous (mean / variance / dependence). Training one
   model per channel and blending, or adding a `t`-bucketed model, sometimes beats a single
   global fit — early steps and late steps are genuinely different problems.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# TEST 1: Per-step AUC Curve (AUC(t) vs t)
def safe_auc(g):
    if g['target'].nunique() > 1:
        return roc_auc_score(g['target'], g['prediction'])
    return np.nan

auc_t = merged.groupby('time_online').apply(safe_auc)

plt.figure(figsize=(10, 5))
plt.plot(auc_t.index, auc_t.values, label='Cross-sectional AUC', color='blue', alpha=0.7)
plt.axhline(0.5, color='red', linestyle='--', label='Random (0.5)')
plt.title('Test 1: Per-Step AUC(t)')
plt.xlabel('Time Step (t)')
plt.ylabel('ROC AUC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# TEST 2: AUC Stratified by Elapsed Time Since Break (d)
# 1. Find tau for each series (first step where target == 1)
tau_df = merged[merged['target'] == 1].groupby('id')['time_online'].min().reset_index().rename(columns={'time_online': 'tau'})
df_d = merged.merge(tau_df, on='id', how='left')

# 2. Calculate age of break 'd'
df_d['d'] = df_d['time_online'] - df_d['tau']

# 3. Vectorized Exact Individual AUC calculation
df_d['rank_all'] = df_d.groupby('time_online')['prediction'].rank()
pos_mask = df_d['target'] == 1
df_d.loc[pos_mask, 'rank_pos'] = df_d[pos_mask].groupby('time_online')['prediction'].rank()

neg_counts = df_d[~pos_mask].groupby('time_online').size()
df_d['n_neg'] = df_d['time_online'].map(neg_counts)

df_d.loc[pos_mask, 'indiv_auc'] = (df_d.loc[pos_mask, 'rank_all'] - df_d.loc[pos_mask, 'rank_pos']) / df_d.loc[pos_mask, 'n_neg']

auc_d = df_d[pos_mask].groupby('d')['indiv_auc'].mean()
auc_d_smooth = auc_d.rolling(window=10, min_periods=1, center=True).mean()

plt.figure(figsize=(10, 5))
plt.plot(auc_d.index, auc_d.values, color='gray', alpha=0.4, label='Raw AUC(d)')
plt.plot(auc_d_smooth.index, auc_d_smooth.values, color='green', linewidth=2.5, label='Smoothed AUC(d)')
plt.axhline(0.5, color='red', linestyle='--')
plt.title('Test 2: AUC Stratified by Age of Break (d = t - τ)')
plt.xlabel('Elapsed Time Since Break (d)')
plt.ylabel('ROC AUC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# TEST 3: Cross-sectional Std of Predictions at Fixed t
std_t = merged.groupby('time_online')['prediction'].std()

plt.figure(figsize=(10, 5))
plt.plot(std_t.index, std_t.values, color='purple')
plt.title('Test 3: Cross-Sectional Prediction Std')
plt.xlabel('Time Step (t)')
plt.ylabel('Standard Deviation of Scores')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
"""
Causal sup-KS scan statistic — corrected diagnostic.

Fully CAUSAL (uses only data up to t, never tau), so its TS-AUC is a real
competing baseline, not a ceiling. If it beats the LightGBM bank on the SAME
held-out series, the bank has a problem.
"""

import glob
import numpy as np

MIN_W = 12          # smallest window length considered
MAX_W = 512         # cap: longer windows cost more, add little power
T_STRIDE = 1        # evaluate every T_STRIDE steps (safe: depends only on t)

_ARANGE = {}


def _idx(m):
    """Cached np.arange(1, m+1) — window lengths only take ~7 distinct values."""
    a = _ARANGE.get(m)
    if a is None:
        a = _ARANGE[m] = np.arange(1, m + 1, dtype=np.float64)
    return a


def _window_lengths(m_max):
    """Geometric ladder of trailing-window lengths, plus the full history so far."""
    ms = []
    m = MIN_W
    while m <= min(m_max, MAX_W):
        ms.append(m)
        m *= 2
    if m_max <= MAX_W and (not ms or ms[-1] != m_max):
        ms.append(m_max)
    if not ms:
        ms = [m_max]
    return ms


def ks_scan_scores(x_hist, x_online):
    """One causal sup-KS score per online step."""
    xh = np.asarray(x_hist, dtype=np.float64).ravel()
    xo = np.asarray(x_online, dtype=np.float64).ravel()
    n = xh.size
    T = xo.size

    # PIT the whole online segment once: u ~ Uniform(0,1) under "no break".
    xs = np.sort(xh)
    u = np.searchsorted(xs, xo, side="right") / float(n)

    out = np.zeros(T, dtype=np.float64)
    last = 0.0

    for t in range(T):
        if t % T_STRIDE != 0:
            out[t] = last
            continue

        m_max = t + 1
        best = 0.0
        for m in _window_lengths(m_max):
            w = np.sort(u[t + 1 - m: t + 1])
            i = _idx(m)
            d_plus = np.max(i / m - w)
            d_minus = np.max(w - (i - 1.0) / m)
            d = d_plus if d_plus > d_minus else d_minus
            en = np.sqrt(n * m / float(n + m))     # two-sample scaling
            v = en * d
            if v > best:
                best = v

        last = best
        out[t] = best

    return out


# ---------------------------------------------------------------------------
# Evaluation helpers
# ---------------------------------------------------------------------------
def evaluate_scan(items, ts_auc, n_series=500, verbose=True):
    ys, ts, ps, ids = [], [], [], []

    for i, (did, xh, xo, tau) in enumerate(list(items)[:n_series]):
        s = ks_scan_scores(xh, xo)
        T = len(s)
        ys.extend(0 if (tau is None or k < tau) else 1 for k in range(T))
        ts.extend(range(T))
        ps.extend(s.tolist())
        ids.extend([did] * T)
        if verbose and (i + 1) % 100 == 0:
            print("  %d/%d series" % (i + 1, n_series))

    auc = ts_auc(np.asarray(ys), np.asarray(ts), np.asarray(ps))
    print("Causal sup-KS scan TS-AUC: %.5f  (%d series)" % (auc, min(n_series, len(list(items)))))
    return auc, np.asarray(ids), np.asarray(ys), np.asarray(ts), np.asarray(ps)


def evaluate_model_same_subset(items, booster, best_it, extract_series,
                               ts_auc, n_series=500):
    ys, ts, ps = [], [], []
    for did, xh, xo, tau in list(items)[:n_series]:
        X, y, tt, _ = extract_series(xh, xo, tau, subsample=False)
        p = booster.predict(X, num_iteration=best_it)
        ys.append(y); ts.append(tt); ps.append(p)
    ys = np.concatenate(ys); ts = np.concatenate(ts); ps = np.concatenate(ps)
    auc = ts_auc(ys, ts, ps)
    print("LightGBM TS-AUC on same subset: %.5f" % auc)
    return auc


# ---------------------------------------------------------------------------
# Run it
# ---------------------------------------------------------------------------
# 1. Recover the trained booster.  It was a LOCAL inside train(); crunch_tools.test()
#    called train(), which dumped it to disk and returned nothing to this namespace.
paths = glob.glob("**/model.joblib", recursive=True)
print("found artefacts:", paths)
art = joblib.load(paths[0])
booster, best_it = art["booster"], art["best_iteration"]

# 2. Reproduce train()'s held-out split EXACTLY (same seed, same VALID_FRACTION).
#    Scoring the model on train_data[:500] would be in-sample; the scan statistic
#    has no fitted parameters, so that comparison would be meaningless.
items = list(train_data)
rng = np.random.default_rng(0)
is_val = rng.random(len(items)) < VALID_FRACTION
va_items = [r for r, v in zip(items, is_val) if v][:500]
print("held-out series:", len(va_items))

# 3. Same series, both detectors.
scan_auc, scan_ids, scan_y, scan_t, scan_p = evaluate_scan(
    va_items, ts_auc, n_series=len(va_items))
model_auc = evaluate_model_same_subset(
    va_items, booster, best_it, extract_series, ts_auc, n_series=len(va_items))

print()
print("sup-KS scan : %.5f" % scan_auc)
print("LightGBM    : %.5f" % model_auc)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_feature_auc_by_d(X, y, t, g, features_to_plot, all_feature_names):
    # 1. Reconstruct a DataFrame with the necessary columns
    df = pd.DataFrame({'id': g, 't': t, 'target': y})
    
    # Map the requested features from the X matrix
    for f in features_to_plot:
        if f in all_feature_names:
            idx = all_feature_names.index(f)
            df[f] = X[:, idx]
        else:
            print(f"Warning: Feature '{f}' not found in the feature list.")
            return

    # 2. Find tau for each series
    tau_df = df[df['target'] == 1].groupby('id')['t'].min().reset_index().rename(columns={'t': 'tau'})
    df_d = df.merge(tau_df, on='id', how='left')
    
    # 3. Calculate age of break 'd'
    df_d['d'] = df_d['t'] - df_d['tau']
    pos_mask = df_d['target'] == 1
    
    # Count negatives at each t
    neg_counts = df_d[~pos_mask].groupby('t').size()
    df_d['n_neg'] = df_d['t'].map(neg_counts)
    
    plt.figure(figsize=(12, 6))
    
    # 4. Calculate and plot AUC(d) for each feature
    for f in features_to_plot:
        # Fractional rank calculation
        df_d['rank_all'] = df_d.groupby('t')[f].rank()
        df_d.loc[pos_mask, 'rank_pos'] = df_d[pos_mask].groupby('t')[f].rank()
        
        # Individual AUC
        df_d.loc[pos_mask, 'indiv_auc'] = (df_d.loc[pos_mask, 'rank_all'] - df_d.loc[pos_mask, 'rank_pos']) / df_d.loc[pos_mask, 'n_neg']
        
        # Group by age 'd'
        auc_d = df_d[pos_mask].groupby('d')['indiv_auc'].mean()
        
        # If the feature is negatively correlated (AUC < 0.5), flip it so signal strength is comparable
        if auc_d.mean() < 0.5:
            auc_d = 1.0 - auc_d
            
        # Smooth the curve for readability
        auc_d_smooth = auc_d.rolling(window=10, min_periods=1, center=True).mean()
        
        plt.plot(auc_d_smooth.index, auc_d_smooth.values, linewidth=2.5, label=f)

    plt.axhline(0.5, color='red', linestyle='--', label='Random (0.5)')
    plt.title('Feature Signal Strength Stratified by Age of Break (d)')
    plt.xlabel('Elapsed Time Since Break (d = t - τ)')
    plt.ylabel('ROC AUC (Smoothed & Direction-Adjusted)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Xd/yd/td/gd/FEATURES_D come from the diagnostic run (section 3) -- a full-registry
# matrix over DIAG_N_SERIES sampled series, so any feature name in FEATURES_D works here,
# not just the original z_* bank. Try one from a family the diagnostic/ablate flagged as
# promising, e.g. an e2_* or lmom_lscale_e_* feature, alongside the old ones for comparison.
features_to_test = ['z_exp', 'z_cusum0.25', 'z_w64']
plot_feature_auc_by_d(Xd, yd, td, gd, features_to_test, FEATURES_D)